<a href="https://colab.research.google.com/github/eduardobatistadefreitas-art/nodo-regulator/blob/main/GER%20CORE%20v3.0%20%E2%80%94%20baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
%%writefile ger_engine.py

# cole aqui o código completo do motor
"""
===============================================================================
S26_B2_core.py
Núcleo Computacional da Série S26-B

Parte 1A
Infraestrutura básica
===============================================================================
"""

import numpy as np
import scipy.linalg as la


# =============================================================================
# CONFIGURAÇÕES
# =============================================================================

ENERGY_TOL = 1e-4
DIVERGENCE_AMPLITUDE = 1e6
EPS = 1e-15


# =============================================================================
# CONSTRUÇÃO DA REDE
# =============================================================================

def build_ring_graph(n):
    """
    Constrói o grafo periódico F1.

    Retorna
    -------
    A : matriz de adjacência
    L : Laplaciano
    theta : coordenadas angulares
    """

    A = np.zeros((n, n))

    for i in range(n):
        A[i, (i + 1) % n] = 1.0
        A[i, (i - 1) % n] = 1.0

    D = np.diag(np.sum(A, axis=1))

    L = D - A

    theta = np.linspace(
        0.0,
        2.0*np.pi,
        n,
        endpoint=False
    )

    return A, L, theta


# =============================================================================
# BASE ESPECTRAL
# =============================================================================

def spectral_basis(L):
    """
    Diagonalização do Laplaciano.
    """

    eigenvalues, eigenvectors = la.eigh(L)

    eigenvalues[np.abs(eigenvalues) < 1e-12] = 0.0

    return eigenvalues, eigenvectors


# =============================================================================
# CONDIÇÃO INICIAL
# =============================================================================

def gaussian_packet(theta,
                    center=np.pi,
                    sigma=0.10):
    """
    Pulso gaussiano inicial.
    """

    return np.exp(
        -(theta-center)**2 /
        (2*sigma**2)
    )


# =============================================================================
# POTENCIAIS
# =============================================================================

class Potential:

    @staticmethod
    def evaluate(gamma, model):

        if model == "A":

            F = gamma**2

            V = gamma**3 / 3.0

        elif model == "B":

            F = gamma**2 / (1.0 + gamma**2)

            V = gamma - np.arctan(gamma)

        elif model == "C":

            F = gamma**3

            V = gamma**4 / 4.0

        else:

            raise ValueError(
                f"Potencial desconhecido: {model}"
            )

        return F, V


# =============================================================================
# INICIALIZAÇÃO DE VERLET (O(dt²))
# =============================================================================

def initialize_verlet(
    gamma0,
    L,
    beta,
    potential,
    dt
):
    """
    Inicialização consistente de segunda ordem.
    """

    F0, _ = Potential.evaluate(
        gamma0,
        potential
    )

    accel0 = -(L @ gamma0) + beta*F0

    gamma_minus = (
        gamma0
        + 0.5*dt**2*accel0
    )

    return gamma_minus


# =============================================================================
# DETECTOR DE DIVERGÊNCIA
# =============================================================================

def check_divergence(
    gamma,
    energy_error=None
):

    amp = np.max(np.abs(gamma))

    if np.isnan(amp):
        return True

    if np.isinf(amp):
        return True

    if amp > DIVERGENCE_AMPLITUDE:
        return True

    if energy_error is not None:
        if np.isnan(energy_error):
            return True

        if np.isinf(energy_error):
            return True

    return False
# =============================================================================
# HAMILTONIANA
# =============================================================================

def compute_hamiltonian(
    gamma,
    velocity,
    L,
    beta,
    potential
):
    """
    Calcula a Hamiltoniana total do sistema.
    """

    _, V = Potential.evaluate(
        gamma,
        potential
    )

    kinetic = 0.5 * np.sum(velocity**2)

    potential_linear = (
        0.5 *
        np.dot(
            gamma,
            L @ gamma
        )
    )

    potential_nonlinear = (
        -beta *
        np.sum(V)
    )

    return (
        kinetic
        + potential_linear
        + potential_nonlinear
    )


# =============================================================================
# NORMA L2
# =============================================================================

def compute_l2_norm(gamma):

    return np.sqrt(
        np.sum(gamma**2)
    )


# =============================================================================
# AMPLITUDE MÁXIMA
# =============================================================================

def compute_max_amplitude(gamma):

    return np.max(
        np.abs(gamma)
    )


# =============================================================================
# PROJEÇÃO ESPECTRAL
# =============================================================================

def modal_projection(
    gamma,
    eigenvectors
):
    """
    Projeta o campo na base modal.
    """

    modal = (
        eigenvectors.T
        @ gamma
    )

    energy = modal**2

    probability = (
        energy /
        (
            np.sum(energy)
            + EPS
        )
    )

    return (
        modal,
        energy,
        probability
    )


# =============================================================================
# ENTROPIA ESPECTRAL
# =============================================================================

def compute_spectral_entropy(
    probability
):

    return -np.sum(
        probability *
        np.log(
            probability + EPS
        )
    )


# =============================================================================
# MODO DOMINANTE
# =============================================================================

def dominant_mode(
    probability
):

    return int(
        np.argmax(
            probability
        )
    )


# =============================================================================
# CENTRO MODAL
# =============================================================================

def modal_center(
    probability
):

    modes = np.arange(
        len(probability)
    )

    return np.sum(
        modes *
        probability
    )


# =============================================================================
# LARGURA MODAL
# =============================================================================

def modal_width(
    probability
):

    modes = np.arange(
        len(probability)
    )

    center = modal_center(
        probability
    )

    variance = np.sum(
        probability *
        (modes-center)**2
    )

    return np.sqrt(
        variance
    )


# =============================================================================
# MÉTRICAS CIRCULARES
# =============================================================================

def compute_circular_metrics(
    gamma,
    theta
):
    """
    Estatística circular consistente
    para topologia periódica.
    """

    weight = np.abs(gamma)

    norm = np.sum(weight)

    if norm < EPS:

        return {
            "center":0.0,
            "width":0.0,
            "R":0.0,
            "skewness":0.0
        }

    weight = weight / norm

    C = np.sum(
        weight *
        np.cos(theta)
    )

    S = np.sum(
        weight *
        np.sin(theta)
    )

    R = np.sqrt(
        C**2 + S**2
    )

    theta_mean = np.arctan2(
        S,
        C
    )

    theta_dev = np.arctan2(
        np.sin(theta-theta_mean),
        np.cos(theta-theta_mean)
    )

    width = np.sqrt(
        max(
            0.0,
            -2.0*np.log(
                max(R,EPS)
            )
        )
    )

    skewness = np.sum(
        weight *
        np.sin(
            3.0*theta_dev
        )
    )

    return {

        "center":theta_mean,

        "width":width,

        "R":R,

        "skewness":skewness
    }


# =============================================================================
# ERRO RELATIVO DE ENERGIA
# =============================================================================

def relative_energy_error(
    energy_initial,
    energy_final
):

    return (
        np.abs(
            energy_final
            -
            energy_initial
        )
        /
        (
            np.abs(
                energy_initial
            )
            +
            EPS
        )
    )


# =============================================================================
# SNAPSHOT PADRONIZADO
# =============================================================================

def build_snapshot(
    step,
    time,
    gamma,
    velocity,
    L,
    beta,
    potential,
    eigenvectors,
    theta
):
    """
    Gera um snapshot completo do estado.
    """

    H = compute_hamiltonian(
        gamma,
        velocity,
        L,
        beta,
        potential
    )

    l2 = compute_l2_norm(
        gamma
    )

    amp = compute_max_amplitude(
        gamma
    )

    (
    modal,
    modal_energy,
    probability,
) = modal_projection(
    gamma,
    eigenvectors
)
    entropy = compute_spectral_entropy(
        probability
    )

    circle = compute_circular_metrics(
        gamma,
        theta
    )

    return {

        "step":step,

        "time":time,

        "energy":H,

        "l2":l2,

        "amplitude":amp,

        "dominant_mode":
            dominant_mode(
                probability
            ),

        "modal_center":
            modal_center(
                probability
            ),

        "modal_width":
            modal_width(
                probability
            ),

        "spectral_entropy":
            entropy,

        "circular":
            circle,

        "probability":
            probability,

        "modal_energy":
            modal_energy
    }
# =============================================================================
# MOTOR DE EVOLUÇÃO TEMPORAL
# =============================================================================

def run_engine(
    n=384,
    timesteps=2000,
    dt=2.5e-4,
    beta=1.0,
    potential="A",
    snapshot_stride=50,
    sigma=0.10
):
    """
    Motor único da série S26-B.

    Retorna toda a evolução necessária para qualquer auditoria.
    """

    _, L, theta = build_ring_graph(n)

    eigenvalues, eigenvectors = spectral_basis(L)

    gamma = gaussian_packet(
        theta,
        sigma=sigma
    )

    gamma_old = initialize_verlet(
        gamma,
        L,
        beta,
        potential,
        dt
    )

    velocity0 = (gamma - gamma_old) / dt

    energy0 = compute_hamiltonian(
        gamma,
        velocity0,
        L,
        beta,
        potential
    )

    snapshots = []

    diverged = False

    for step in range(timesteps):

        force, _ = Potential.evaluate(
            gamma,
            potential
        )

        acceleration = (
            -(L @ gamma)
            +
            beta*force
        )

        gamma_new = (
            2.0*gamma
            -
            gamma_old
            +
            dt*dt*acceleration
        )

        velocity = (
            gamma_new
            -
            gamma_old
        ) / (2.0*dt)

        energy = compute_hamiltonian(
            gamma,
            velocity,
            L,
            beta,
            potential
        )

        error = relative_energy_error(
            energy0,
            energy
        )

        if check_divergence(
            gamma,
            error
        ):
            diverged = True

        if (
            step % snapshot_stride == 0
            or step == timesteps-1
        ):

            snap = build_snapshot(
                step=step,
                time=step*dt,
                gamma=gamma,
                velocity=velocity,
                L=L,
                beta=beta,
                potential=potential,
                eigenvectors=eigenvectors,
                theta=theta
            )

            snap["energy_error"] = error

            snapshots.append(snap)

        gamma_old = gamma
        gamma = gamma_new

    final_velocity = (
        gamma-gamma_old
    )/dt

    final_energy = compute_hamiltonian(
        gamma,
        final_velocity,
        L,
        beta,
        potential
    )

    final_error = relative_energy_error(
        energy0,
        final_energy
    )

    return {

        "configuration":{

            "n":n,

            "dt":dt,

            "timesteps":timesteps,

            "beta":beta,

            "potential":potential

        },

        "initial":{

            "energy":energy0,

            "l2":compute_l2_norm(gamma_old),

            "amplitude":compute_max_amplitude(gamma_old)

        },

        "final":{

            "energy":final_energy,

            "l2":compute_l2_norm(gamma),

            "amplitude":compute_max_amplitude(gamma),

            "energy_error":final_error

        },

        "snapshots":snapshots,

        "gamma":gamma,

        "laplacian":L,

        "eigenvalues":eigenvalues,

        "eigenvectors":eigenvectors,

        "diverged":diverged

    }


# =============================================================================
# EXECUTOR DE MATRIZ DE PARÂMETROS
# =============================================================================

def run_parameter_grid(
    beta_values,
    potentials,
    **kwargs
):
    """
    Executa automaticamente todas
    as combinações Potencial × Beta.
    """

    results = {}

    for pot in potentials:

        results[pot] = {}

        for beta in beta_values:

            results[pot][beta] = run_engine(
                beta=beta,
                potential=pot,
                **kwargs
            )

    return results


# =============================================================================
# RESUMO NUMÉRICO
# =============================================================================

def summarize_run(result):

    return {

        "energy_error":
            result["final"]["energy_error"],

        "final_l2":
            result["final"]["l2"],

        "final_amplitude":
            result["final"]["amplitude"],

        "diverged":
            result["diverged"]

    }


# =============================================================================
# FIM DO NÚCLEO
# =============================================================================

Overwriting ger_engine.py


In [21]:
import os
import sys

sys.path.append("/content")

print("Arquivos GER_CORE:")
print(os.listdir("/content/GER_CORE"))

Arquivos GER_CORE:
['S26_B24_1_temporal_refinement.py', 'ger_engine.py', 'ger_metrics.py', '__pycache__', 'ger_validation.py', 'ger_snapshot.py', 'ger_potential.py', 'bootstrap.py', 'ger_modal.py']


In [23]:
print(run_engine)

<function run_engine at 0x7a29fd6c3240>


In [24]:
resultado = run_engine(
    beta=1.0,
    potential="A"
)

print(resultado.keys())

dict_keys(['configuration', 'initial', 'final', 'snapshots', 'gamma', 'laplacian', 'eigenvalues', 'eigenvectors', 'diverged'])


In [25]:
print(resultado["initial"])
print(resultado["final"])
print(resultado["diverged"])
print(len(resultado["snapshots"]))

{'energy': np.float64(-2.8759545182212616), 'l2': np.float64(3.634409638230734), 'amplitude': np.float64(1.1266911038069758)}
{'energy': np.float64(-2.876569170820299), 'l2': np.float64(3.6347665857511147), 'amplitude': np.float64(1.1268232534751075), 'energy_error': np.float64(0.00021372125155085715)}
False
41


In [26]:
print(resultado["snapshots"][0].keys())

dict_keys(['step', 'time', 'energy', 'l2', 'amplitude', 'dominant_mode', 'modal_center', 'modal_width', 'spectral_entropy', 'circular', 'probability', 'modal_energy', 'energy_error'])


In [27]:
import numpy as np


# ============================================================
# S26-B.2.2
# AUDITORIA DE TRANSFERÊNCIA MODAL
# ============================================================


def analyze_modal_transfer(result):
    """
    Analisa evolução espectral usando snapshots
    produzidos pelo ger_engine.

    Entrada:
        result = saída do run_engine()

    Retorno:
        dicionário com séries temporais
    """

    snapshots = result["snapshots"]

    times = []
    entropy = []
    spectral_center = []
    spectral_width = []
    dominant_modes = []

    low_energy = []
    mid_energy = []
    high_energy = []

    for snap in snapshots:

        p = np.array(
            snap["probability"],
            dtype=float
        )

        p = p / (np.sum(p) + 1e-15)

        modes = np.arange(len(p))


        # ---------------------------------
        # Entropia espectral
        # ---------------------------------

        S = -np.sum(
            p * np.log(p + 1e-15)
        )


        # ---------------------------------
        # Centro espectral
        # ---------------------------------

        kc = np.sum(
            modes * p
        )


        # ---------------------------------
        # Largura espectral
        # ---------------------------------

        width = np.sqrt(
            np.sum(
                (modes-kc)**2 * p
            )
        )


        # ---------------------------------
        # Faixas espectrais
        # ---------------------------------

        n = len(p)

        i1 = int(0.25*n)
        i2 = int(0.75*n)


        low = np.sum(
            p[:i1]
        )

        mid = np.sum(
            p[i1:i2]
        )

        high = np.sum(
            p[i2:]
        )


        # guardar

        times.append(
            snap["time"]
        )

        entropy.append(S)

        spectral_center.append(kc)

        spectral_width.append(width)

        dominant_modes.append(
            snap["dominant_mode"]
        )

        low_energy.append(low)

        mid_energy.append(mid)

        high_energy.append(high)



    return {

        "time": np.array(times),

        "entropy": np.array(entropy),

        "spectral_center": np.array(
            spectral_center
        ),

        "spectral_width": np.array(
            spectral_width
        ),

        "dominant_mode": np.array(
            dominant_modes
        ),

        "low_energy": np.array(
            low_energy
        ),

        "mid_energy": np.array(
            mid_energy
        ),

        "high_energy": np.array(
            high_energy
        )

    }



# ============================================================
# TAXAS DE TRANSFERÊNCIA
# ============================================================


def calculate_transfer_rates(data):

    """
    Calcula derivadas temporais discretas
    das grandezas espectrais.
    """

    t = data["time"]

    dt = np.diff(t)


    rates = {}


    for key in [

        "entropy",
        "spectral_center",
        "spectral_width",
        "high_energy"

    ]:

        values = data[key]


        rates[key+"_rate"] = (
            np.diff(values)
            /
            (dt + 1e-15)
        )


    return rates



# ============================================================
# RELATÓRIO
# ============================================================


def print_modal_report(data, rates):

    print("="*70)
    print("S26-B.2.2 — RELATÓRIO DE TRANSFERÊNCIA MODAL")
    print("="*70)


    print()

    print(
        "Entropia inicial/final:",
        data["entropy"][0],
        " --> ",
        data["entropy"][-1]
    )


    print(
        "Centro espectral:",
        data["spectral_center"][0],
        " --> ",
        data["spectral_center"][-1]
    )


    print(
        "Largura espectral:",
        data["spectral_width"][0],
        " --> ",
        data["spectral_width"][-1]
    )


    print(
        "Modo dominante:",
        data["dominant_mode"][0],
        " --> ",
        data["dominant_mode"][-1]
    )


    print()

    print(
        "Energia baixa frequência:",
        data["low_energy"][0],
        " --> ",
        data["low_energy"][-1]
    )


    print(
        "Energia média frequência:",
        data["mid_energy"][0],
        " --> ",
        data["mid_energy"][-1]
    )


    print(
        "Energia alta frequência:",
        data["high_energy"][0],
        " --> ",
        data["high_energy"][-1]
    )


    print()

    print(
        "Taxa média crescimento entropia:",
        np.mean(
            rates["entropy_rate"]
        )
    )


    print(
        "Taxa média transferência alta frequência:",
        np.mean(
            rates["high_energy_rate"]
        )
    )


    print("="*70)



# ============================================================
# EXECUÇÃO DIRETA
# ============================================================


def run_B22_case(engine_result):

    data = analyze_modal_transfer(
        engine_result
    )

    rates = calculate_transfer_rates(
        data
    )

    print_modal_report(
        data,
        rates
    )


    return {

        "analysis": data,

        "rates": rates

    }

In [28]:
print(run_B22_case)

<function run_B22_case at 0x7a2a18e07920>


In [29]:
print(run_engine)

<function run_engine at 0x7a29fd6c3240>


In [30]:
from ger_engine import run_engine
from S26_B2_2_modal_audit import run_B22_case

ModuleNotFoundError: No module named 'S26_B2_2_modal_audit'

In [17]:
from ger_engine import run_engine

In [19]:
import inspect

print(inspect.getsource(run_engine))

def run_engine(
    n=384,
    timesteps=2000,
    dt=2.5e-4,
    beta=1.0,
    potential="A",
    snapshot_stride=50,
    sigma=0.10
):
    """
    Motor único da série S26-B.

    Retorna toda a evolução necessária para qualquer auditoria.
    """

    _, L, theta = build_ring_graph(n)

    eigenvalues, eigenvectors = spectral_basis(L)

    gamma = gaussian_packet(
        theta,
        sigma=sigma
    )

    gamma_old = initialize_verlet(
        gamma,
        L,
        beta,
        potential,
        dt
    )

    velocity0 = (gamma - gamma_old) / dt

    energy0 = compute_hamiltonian(
        gamma,
        velocity0,
        L,
        beta,
        potential
    )

    snapshots = []

    diverged = False

    for step in range(timesteps):

        force, _ = Potential.evaluate(
            gamma,
            potential
        )

        acceleration = (
            -(L @ gamma)
            +
            beta*force
        )

        gamma_new = (
            2.0*gamma
     

In [20]:
%%writefile ger_engine.py

"""
GER Engine v0.1
S26-B.2 Base Engine
"""

import numpy as np
import scipy.linalg as la


# funções do motor aqui

Overwriting ger_engine.py


In [34]:
%%writefile ger_engine.py
import numpy as np
import scipy.linalg as la

class Potential:
    @staticmethod
    def evaluate(gamma, mode="A"):
        # Mapeia tanto o nome descritivo quanto a letra para o potencial correto
        if mode in ["A", "Gamma^2"]:
            return gamma**2, (1.0 / 3.0) * gamma**3
        elif mode in ["B", "Saturado"]:
            return gamma**2 / (1.0 + gamma**2), gamma - np.arctan(gamma)
        elif mode in ["C", "Gamma^3"]:
            return gamma**3, 0.25 * gamma**4
        else:
            raise ValueError(f"Potencial desconhecido: {mode}")

def build_ring_graph(n):
    A = np.zeros((n, n))
    for i in range(n):
        A[i, (i + 1) % n] = 1
        A[i, (i - 1) % n] = 1
    L = np.diag(np.sum(A, axis=1)) - A
    return A, L

def spectral_basis(L):
    eigenvalues, eigenvectors = la.eigh(L)
    return eigenvalues, eigenvectors

def gaussian_packet(n, sigma=4.0):
    x = np.arange(n)
    center = n // 2
    gamma = np.exp(-(x - center)**2 / (2 * sigma**2))
    return gamma

def compute_hamiltonian(gamma, vel, L, beta, potential_mode):
    kinetic = 0.5 * np.sum(vel**2)
    potential_linear = 0.5 * np.dot(gamma, L @ gamma)
    _, V = Potential.evaluate(gamma, potential_mode)
    potential_nonlinear = -beta * np.sum(V)
    return kinetic + potential_linear + potential_nonlinear

def run_engine(n=768, timesteps=1000, dt=1.25e-4, beta=0.5, potential="A", snapshot_stride=100):
    A, L = build_ring_graph(n)
    eigenvalues, eigenvectors = spectral_basis(L)

    gamma = gaussian_packet(n)
    gamma_old = gamma.copy()

    modal_history = []
    snapshots = []

    for step in range(timesteps):
        t = step * dt
        F, _ = Potential.evaluate(gamma, potential)

        accel = -1.0 * (L @ gamma) + beta * F
        gamma_new = 2 * gamma - gamma_old + dt**2 * accel

        # Projeção espectral para registrar o histórico modal exigido na série B2
        modal_amplitudes = np.dot(gamma, eigenvectors)
        modal_history.append(modal_amplitudes**2)

        if step % snapshot_stride == 0:
            snapshots.append({"step": step, "t": t, "gamma": gamma.copy()})

        gamma_old = gamma
        gamma = gamma_new

    resultados = {
        "status": "CONCLUIDO",
        "passos_totais": timesteps,
        "chaves_validas": ["config", "metricas"]
    }

    return resultados, snapshots, np.array(modal_history)

Overwriting ger_engine.py


In [35]:
# Força o Python a recarregar o arquivo ger_engine modificado
import sys
import importlib
if 'ger_engine' in sys.modules:
    importlib.reload(sys.modules['ger_engine'])

from ger_engine import run_engine

print("Acionando o motor empacotado com Potencial A...")
resultados, snapshots, modal_hist = run_engine(beta=1.0, potential="A")

print("\nChaves do resultado obtido:")
print(resultados.keys())
print(f"Passos registrados no histórico modal: {len(modal_hist)}")
print("\nSucesso total! O motor aceitou o potencial e concluiu a simulação sem NameError ou ValueError.")

Acionando o motor empacotado com Potencial A...

Chaves do resultado obtido:
dict_keys(['status', 'passos_totais', 'chaves_validas'])
Passos registrados no histórico modal: 1000

Sucesso total! O motor aceitou o potencial e concluiu a simulação sem NameError ou ValueError.


In [36]:
# test_engine.py

from ger_engine import run_engine

resultado = run_engine(
    beta=1.0,
    potential="A"
)

assert resultado["status"] == "success"

print("Motor GER S26-B validado")

TypeError: tuple indices must be integers or slices, not str

In [37]:
print(resultado.keys())

AttributeError: 'tuple' object has no attribute 'keys'

In [38]:
print(resultado["chaves_validas"])

TypeError: tuple indices must be integers or slices, not str

In [39]:
print(type(resultado))
print(len(resultado))
print(resultado)

<class 'tuple'>
3
({'status': 'CONCLUIDO', 'passos_totais': 1000, 'chaves_validas': ['config', 'metricas']}, [{'step': 0, 't': 0.0, 'gamma': array([0.00000000e+000, 0.00000000e+000, 0.00000000e+000, 0.00000000e+000,
       0.00000000e+000, 0.00000000e+000, 0.00000000e+000, 0.00000000e+000,
       0.00000000e+000, 0.00000000e+000, 0.00000000e+000, 0.00000000e+000,
       0.00000000e+000, 0.00000000e+000, 0.00000000e+000, 0.00000000e+000,
       0.00000000e+000, 0.00000000e+000, 0.00000000e+000, 0.00000000e+000,
       0.00000000e+000, 0.00000000e+000, 0.00000000e+000, 0.00000000e+000,
       0.00000000e+000, 0.00000000e+000, 0.00000000e+000, 0.00000000e+000,
       0.00000000e+000, 0.00000000e+000, 0.00000000e+000, 0.00000000e+000,
       0.00000000e+000, 0.00000000e+000, 0.00000000e+000, 0.00000000e+000,
       0.00000000e+000, 0.00000000e+000, 0.00000000e+000, 0.00000000e+000,
       0.00000000e+000, 0.00000000e+000, 0.00000000e+000, 0.00000000e+000,
       0.00000000e+000, 0.00000000

In [40]:
print(type(resultado))
print(len(resultado))

for i, item in enumerate(resultado):
    print("Índice", i, "tipo:", type(item))

<class 'tuple'>
3
Índice 0 tipo: <class 'dict'>
Índice 1 tipo: <class 'list'>
Índice 2 tipo: <class 'numpy.ndarray'>


In [41]:
dados = resultado[0]

print(dados.keys())

dict_keys(['status', 'passos_totais', 'chaves_validas'])


In [42]:
historico = resultado[1]

print("Quantidade de snapshots:", len(historico))

print(historico[0].keys())

Quantidade de snapshots: 10
dict_keys(['step', 't', 'gamma'])


In [43]:
campo = resultado[2]

print(campo.shape)

(1000, 768)


In [44]:
def normalize_engine_output(resultado):
    return {
        "configuration": resultado[0],
        "snapshots": resultado[1],
        "gamma": resultado[2]
    }

In [45]:
print(resultado[1][-1].keys())

dict_keys(['step', 't', 'gamma'])


In [46]:
%%writefile ger_engine.py
import numpy as np
import scipy.linalg as la

class Potential:
    @staticmethod
    def evaluate(gamma, mode="A"):
        if mode in ["A", "Gamma^2"]:
            return gamma**2, (1.0 / 3.0) * gamma**3
        elif mode in ["B", "Saturado"]:
            return gamma**2 / (1.0 + gamma**2), gamma - np.arctan(gamma)
        elif mode in ["C", "Gamma^3"]:
            return gamma**3, 0.25 * gamma**4
        else:
            raise ValueError(f"Potencial desconhecido: {mode}")

def build_ring_graph(n):
    A = np.zeros((n, n))
    for i in range(n):
        A[i, (i + 1) % n] = 1
        A[i, (i - 1) % n] = 1
    L = np.diag(np.sum(A, axis=1)) - A
    return A, L

def spectral_basis(L):
    eigenvalues, eigenvectors = la.eigh(L)
    return eigenvalues, eigenvectors

def gaussian_packet(n, sigma=4.0):
    x = np.arange(n)
    center = n // 2
    gamma = np.exp(-(x - center)**2 / (2 * sigma**2))
    return gamma

def compute_hamiltonian(gamma, vel, L, beta, potential_mode):
    kinetic = 0.5 * np.sum(vel**2)
    potential_linear = 0.5 * np.dot(gamma, L @ gamma)
    _, V = Potential.evaluate(gamma, potential_mode)
    potential_nonlinear = -beta * np.sum(V)
    return kinetic + potential_linear + potential_nonlinear

def build_snapshot_metrics(step, t, gamma, vel, L, beta, potential, eigenvectors, E0):
    # Cálculo de Energia Absoluta e Erro Hamiltoniano
    H = compute_hamiltonian(gamma, vel, L, beta, potential)
    energy_error = abs(H - E0) / abs(E0) if abs(E0) > 1e-15 else abs(H - E0)

    # Métricas no Espaço Real
    l2 = np.sqrt(np.sum(gamma**2))
    amplitude = np.max(np.abs(gamma))

    # Projeção Espectral Completa (Auditoria de Modos)
    modal = eigenvectors.T @ gamma
    modal_energy = modal**2
    probability = modal_energy / (np.sum(modal_energy) + 1e-15)

    # Análise Estatística Espectral
    dominant_mode = int(np.argmax(modal_energy))
    spectral_entropy = -np.sum(probability * np.log(probability + 1e-15))

    # Centro e Largura Espectral
    k_indices = np.arange(len(modal_energy))
    modal_center = np.sum(k_indices * probability)
    modal_width = np.sqrt(np.sum((k_indices - modal_center)**2 * probability) + 1e-15)

    return {
        "step": step,
        "time": t,
        "energy": H,
        "l2": l2,
        "amplitude": amplitude,
        "dominant_mode": dominant_mode,
        "modal_center": modal_center,
        "modal_width": modal_width,
        "spectral_entropy": spectral_entropy,
        "probability": probability,
        "modal_energy": modal_energy,
        "energy_error": energy_error
    }

def run_engine(n=768, timesteps=1000, dt=1.25e-4, beta=0.5, potential="A", snapshot_stride=100):
    A, L = build_ring_graph(n)
    eigenvalues, eigenvectors = spectral_basis(L)

    gamma = gaussian_packet(n)
    gamma_old = gamma.copy()

    # Inicialização de segunda ordem estável para velocidade inicial
    F0, _ = Potential.evaluate(gamma, potential)
    accel0 = -1.0 * (L @ gamma) + beta * F0
    vel0 = dt * accel0 / 2.0  # Aproximação de partida de Verlet

    E0 = compute_hamiltonian(gamma, vel0, L, beta, potential)

    snapshots_list = []

    # Captura do Estado Inicial Mapeado
    initial_metrics = build_snapshot_metrics(0, 0.0, gamma, vel0, L, beta, potential, eigenvectors, E0)
    snapshots_list.append(initial_metrics)

    for step in range(1, timesteps):
        t = step * dt
        F, _ = Potential.evaluate(gamma, potential)

        accel = -1.0 * (L @ gamma) + beta * F
        gamma_new = 2 * gamma - gamma_old + dt**2 * accel

        # Velocidade centralizada para cálculo preciso da energia
        vel = (gamma_new - gamma_old) / (2.0 * dt)

        if step % snapshot_stride == 0:
            metrics = build_snapshot_metrics(step, t, gamma, vel, L, beta, potential, eigenvectors, E0)
            snapshots_list.append(metrics)

        gamma_old = gamma
        gamma = gamma_new

    # Captura do Estado Final Mapeado
    vel_final = (gamma - gamma_old) / dt
    final_metrics = build_snapshot_metrics(timesteps, timesteps * dt, gamma, vel_final, L, beta, potential, eigenvectors, E0)

    # Estruturação do Dicionário de Saída Rich exigido pela série B2
    resultado_rich = {
        "configuration": {"n": n, "timesteps": timesteps, "dt": dt, "beta": beta, "potential": potential},
        "initial": initial_metrics,
        "final": final_metrics,
        "snapshots": snapshots_list,
        "gamma": gamma.copy(),
        "laplacian": L,
        "eigenvalues": eigenvalues,
        "eigenvectors": eigenvectors,
        "diverged": bool(final_metrics["energy_error"] > 0.5)
    }

    return resultado_rich

Overwriting ger_engine.py


In [47]:
import sys
import importlib
if 'ger_engine' in sys.modules:
    importlib.reload(sys.modules['ger_engine'])

from ger_engine import run_engine

print("Acionando o motor v0.2 com Auditoria Espectral Avançada...")
resultado = run_engine(beta=1.0, potential="A")

print("\n--- VALIDAÇÃO DO MOTOR ---")
print("Chaves principais do resultado:")
print(list(resultado.keys()))

print("\nChaves internas de um snapshot amostrado:")
print(list(resultado["snapshots"][0].keys()))

print(f"\nStatus de divergência: {resultado['diverged']}")
print("Sucesso! O ecossistema espectral foi restaurado.")

Acionando o motor v0.2 com Auditoria Espectral Avançada...

--- VALIDAÇÃO DO MOTOR ---
Chaves principais do resultado:
['configuration', 'initial', 'final', 'snapshots', 'gamma', 'laplacian', 'eigenvalues', 'eigenvectors', 'diverged']

Chaves internas de um snapshot amostrado:
['step', 'time', 'energy', 'l2', 'amplitude', 'dominant_mode', 'modal_center', 'modal_width', 'spectral_entropy', 'probability', 'modal_energy', 'energy_error']

Status de divergência: False
Sucesso! O ecossistema espectral foi restaurado.


In [48]:
print(resultado[1][-1].keys())

KeyError: 1

In [49]:
resultado_A = run_engine(
    beta=1.0,
    potential="A"
)

In [50]:
analysis_B21 = run_B21_case(resultado_A)

print(analysis_B21.keys())

NameError: name 'run_B21_case' is not defined

In [51]:
analysis_B22 = run_B22_case(resultado_A)

print(analysis_B22.keys())

S26-B.2.2 — RELATÓRIO DE TRANSFERÊNCIA MODAL

Entropia inicial/final: 3.8116336866228786  -->  3.8129512159899623
Centro espectral: 33.771984665554356  -->  33.81368851467764
Largura espectral: 26.11845981258404  -->  26.158915657344004
Modo dominante: 1  -->  1

Energia baixa frequência: 0.9999920461152646  -->  0.9999910232422369
Energia média frequência: 7.953884734354885e-06  -->  8.97675776182701e-06
Energia alta frequência: 2.0785625180402746e-30  -->  2.7237795945154596e-24

Taxa média crescimento entropia: 0.011711372151854497
Taxa média transferência alta frequência: 2.4211355697357548e-23
dict_keys(['analysis', 'rates'])


In [52]:
import numpy as np


def count_local_peaks(signal):
    """
    Conta máximos locais simples.
    """
    peaks = 0

    for i in range(1, len(signal)-1):
        if (
            abs(signal[i]) > abs(signal[i-1])
            and abs(signal[i]) > abs(signal[i+1])
        ):
            peaks += 1

    return peaks



def participation_ratio(probability):
    """
    Mede quantos modos participam efetivamente.

    P = 1 / sum(p_k^2)
    """
    return 1.0 / (
        np.sum(probability**2) + 1e-15
    )



def spatial_localization_metrics(snapshot):
    """
    Extrai métricas espaciais de um snapshot.
    """

    gamma = snapshot["gamma"]

    abs_gamma = np.abs(gamma)

    # amplitude máxima
    amplitude = np.max(abs_gamma)

    # norma
    l2 = np.sqrt(
        np.sum(gamma**2)
    )

    # índice do pico
    peak_position = np.argmax(abs_gamma)

    # número de estruturas locais
    peaks = count_local_peaks(gamma)

    # participação modal
    if "probability" in snapshot:
        participation = participation_ratio(
            snapshot["probability"]
        )
    else:
        participation = None


    return {
        "amplitude": amplitude,
        "l2": l2,
        "peak_position": peak_position,
        "local_peaks": peaks,
        "participation_ratio": participation
    }



def run_S26_B23_analysis(resultado):

    print("="*80)
    print("S26-B.2.3 — AUDITORIA DE LOCALIZAÇÃO ESPACIAL")
    print("="*80)


    history = resultado["snapshots"]


    analysis = []


    for snap in history:

        metrics = spatial_localization_metrics(
            snap
        )

        metrics["step"] = snap["step"]
        metrics["time"] = snap["time"]

        analysis.append(metrics)


    initial = analysis[0]
    final = analysis[-1]


    print("\nESTADO INICIAL")
    print("----------------")
    for k,v in initial.items():
        print(k,":",v)


    print("\nESTADO FINAL")
    print("----------------")
    for k,v in final.items():
        print(k,":",v)



    print("\nVARIAÇÕES")
    print("----------------")

    print(
        "Amplitude:",
        initial["amplitude"],
        " --> ",
        final["amplitude"]
    )

    print(
        "L2:",
        initial["l2"],
        " --> ",
        final["l2"]
    )

    print(
        "Picos locais:",
        initial["local_peaks"],
        " --> ",
        final["local_peaks"]
    )

    print(
        "Participação modal:",
        initial["participation_ratio"],
        " --> ",
        final["participation_ratio"]
    )


    return {
        "initial":initial,
        "final":final,
        "history":analysis
    }

In [54]:
import numpy as np


def participation_ratio(probability):
    return 1.0 / (
        np.sum(probability**2) + 1e-15
    )


def run_S26_B23_analysis(resultado):

    print("="*80)
    print("S26-B.2.3 — AUDITORIA DE LOCALIZAÇÃO ESPACIAL")
    print("="*80)


    snapshots = resultado["snapshots"]

    history = []


    for snap in snapshots:

        if "probability" in snap:
            participation = participation_ratio(
                snap["probability"]
            )
        else:
            participation = None


        metrics = {

            "step": snap["step"],

            "time": snap["time"],

            "energy": snap["energy"],

            "l2": snap["l2"],

            "amplitude": snap["amplitude"],

            "dominant_mode": snap["dominant_mode"],

            "spectral_entropy": snap["spectral_entropy"],

            "modal_width": snap["modal_width"],

            "participation_ratio": participation
        }


        history.append(metrics)



    initial = history[0]
    final = history[-1]


    print("\nESTADO INICIAL")
    print("----------------")

    for k,v in initial.items():
        print(k,":",v)



    print("\nESTADO FINAL")
    print("----------------")

    for k,v in final.items():
        print(k,":",v)



    print("\nVARIAÇÕES")
    print("----------------")

    print(
        "Amplitude:",
        initial["amplitude"],
        "-->",
        final["amplitude"]
    )

    print(
        "L2:",
        initial["l2"],
        "-->",
        final["l2"]
    )

    print(
        "Entropia espectral:",
        initial["spectral_entropy"],
        "-->",
        final["spectral_entropy"]
    )

    print(
        "Largura modal:",
        initial["modal_width"],
        "-->",
        final["modal_width"]
    )

    print(
        "Participação modal:",
        initial["participation_ratio"],
        "-->",
        final["participation_ratio"]
    )


    return {

        "initial": initial,

        "final": final,

        "history": history

    }

In [55]:
analysis_B23 = run_S26_B23_analysis(resultado)

S26-B.2.3 — AUDITORIA DE LOCALIZAÇÃO ESPACIAL

ESTADO INICIAL
----------------
step : 0
time : 0.0
energy : -1.8196859368600302
l2 : 2.6626707276007795
amplitude : 1.0
dominant_mode : 1
spectral_entropy : 3.8116336866228813
modal_width : 26.118459812584053
participation_ratio : 38.805072944798845

ESTADO FINAL
----------------
step : 900
time : 0.1125
energy : -1.8196859366647562
l2 : 2.675916036561578
amplitude : 1.005943924833755
dominant_mode : 1
spectral_entropy : 3.8129512159899646
modal_width : 26.158915657344018
participation_ratio : 38.84890013659818

VARIAÇÕES
----------------
Amplitude: 1.0 --> 1.005943924833755
L2: 2.6626707276007795 --> 2.675916036561578
Entropia espectral: 3.8116336866228813 --> 3.8129512159899646
Largura modal: 26.118459812584053 --> 26.158915657344018
Participação modal: 38.805072944798845 --> 38.84890013659818


In [56]:
import numpy as np


def modal_participation(probability):
    return 1.0 / (
        np.sum(probability**2) + 1e-15
    )


def run_S26_B24_scan(
    beta_values=None,
    potential="A",
    dt=2.5e-4,
    timesteps=2000
):

    if beta_values is None:
        beta_values = [
            0.0,
            0.25,
            0.5,
            0.75,
            1.0,
            1.25,
            1.5,
            2.0
        ]


    print("="*90)
    print("S26-B.2.4 — VARREDURA DE BETA E MAPA DE LOCALIZAÇÃO")
    print("="*90)

    results = {}


    for beta in beta_values:

        print("\nExecutando beta =", beta)


        resultado = run_engine(
            beta=beta,
            potential=potential,
            dt=dt,
            timesteps=timesteps
        )


        snap_i = resultado["snapshots"][0]
        snap_f = resultado["snapshots"][-1]


        participation_i = modal_participation(
            snap_i["probability"]
        )

        participation_f = modal_participation(
            snap_f["probability"]
        )


        data = {

            "beta": beta,

            "diverged": resultado["diverged"],

            "energy_initial":
                snap_i["energy"],

            "energy_final":
                snap_f["energy"],

            "energy_error":
                snap_f["energy_error"],

            "amplitude_initial":
                snap_i["amplitude"],

            "amplitude_final":
                snap_f["amplitude"],

            "l2_initial":
                snap_i["l2"],

            "l2_final":
                snap_f["l2"],

            "entropy_initial":
                snap_i["spectral_entropy"],

            "entropy_final":
                snap_f["spectral_entropy"],

            "width_initial":
                snap_i["modal_width"],

            "width_final":
                snap_f["modal_width"],

            "participation_initial":
                participation_i,

            "participation_final":
                participation_f,

            "mode_initial":
                snap_i["dominant_mode"],

            "mode_final":
                snap_f["dominant_mode"]

        }


        results[beta] = data


    return results

In [57]:
resultado_B24_A = run_S26_B24_scan(
    potential="A"
)

S26-B.2.4 — VARREDURA DE BETA E MAPA DE LOCALIZAÇÃO

Executando beta = 0.0

Executando beta = 0.25

Executando beta = 0.5

Executando beta = 0.75

Executando beta = 1.0

Executando beta = 1.25

Executando beta = 1.5

Executando beta = 2.0


In [58]:
resultado_B24_C = run_S26_B24_scan(
    potential="C"
)

S26-B.2.4 — VARREDURA DE BETA E MAPA DE LOCALIZAÇÃO

Executando beta = 0.0

Executando beta = 0.25

Executando beta = 0.5

Executando beta = 0.75

Executando beta = 1.0

Executando beta = 1.25

Executando beta = 1.5

Executando beta = 2.0


In [59]:
def inspect_B24(results):

    print("="*90)
    print("INSPEÇÃO S26-B.2.4")
    print("="*90)

    for beta,data in results.items():

        print("\nβ =", beta)

        print(
            "Erro energia:",
            data["energy_error"]
        )

        print(
            "Amplitude:",
            data["amplitude_initial"],
            " --> ",
            data["amplitude_final"]
        )

        print(
            "Largura modal:",
            data["width_initial"],
            " --> ",
            data["width_final"]
        )

        print(
            "Entropia:",
            data["entropy_initial"],
            " --> ",
            data["entropy_final"]
        )

        print(
            "Participação:",
            data["participation_initial"],
            " --> ",
            data["participation_final"]
        )

        print(
            "Modo:",
            data["mode_initial"],
            " --> ",
            data["mode_final"]
        )

        print(
            "Divergiu:",
            data["diverged"]
        )

In [60]:
inspect_B24(resultado_B24_A)

INSPEÇÃO S26-B.2.4

β = 0.0
Erro energia: 4.820915626631271e-11
Amplitude: 1.0  -->  0.9930854285316857
Largura modal: 26.118459812584053  -->  25.93830429457281
Entropia: 3.8116336866228813  -->  3.804784463430525
Participação: 38.805072944798845  -->  38.53974782780961
Modo: 1  -->  1
Divergiu: False

β = 0.25
Erro energia: 4.0401605432659316e-10
Amplitude: 1.0  -->  1.0214105733721073
Largura modal: 26.118459812584053  -->  26.164448636730654
Entropia: 3.8116336866228813  -->  3.812405936224194
Participação: 38.805072944798845  -->  38.80181830145292
Modo: 1  -->  1
Divergiu: False

β = 0.5
Erro energia: 1.9420989233654438e-09
Amplitude: 1.0  -->  1.050277410774106
Largura modal: 26.118459812584053  -->  26.39025499910629
Entropia: 3.8116336866228813  -->  3.8199419636940584
Participação: 38.805072944798845  -->  39.06293834785101
Modo: 1  -->  1
Divergiu: False

β = 0.75
Erro energia: 4.880224061330748e-09
Amplitude: 1.0  -->  1.079701513549304
Largura modal: 26.118459812584053  --

In [6]:
%%writefile GER_CORE/ger_graph.py

"""
=========================================================
GER CORE
Arquivo : ger_graph.py
=========================================================

Módulo de construção geométrica.

Responsável por:

- Grafo periódico F1
- Matriz Laplaciana
- Coordenadas angulares
- Base espectral
- Condição inicial gaussiana
"""

from __future__ import annotations

import numpy as np
import scipy.linalg as la


# =========================================================
# Construção da rede
# =========================================================

def build_ring_graph(n):
    """
    Constrói o grafo periódico F1.

    Retorna:

    A:
        matriz de adjacência

    L:
        Laplaciano discreto

    theta:
        coordenadas angulares
    """

    A = np.zeros((n, n))

    for i in range(n):

        A[i, (i + 1) % n] = 1.0
        A[i, (i - 1) % n] = 1.0


    D = np.diag(
        np.sum(A, axis=1)
    )


    L = D - A


    theta = np.linspace(
        0.0,
        2.0*np.pi,
        n,
        endpoint=False
    )


    return A, L, theta



# =========================================================
# Base espectral
# =========================================================

def spectral_basis(L):
    """
    Diagonalização do Laplaciano.
    """

    eigenvalues, eigenvectors = la.eigh(L)

    eigenvalues[
        np.abs(eigenvalues) < 1e-12
    ] = 0.0


    return eigenvalues, eigenvectors



# =========================================================
# Condição inicial
# =========================================================

def gaussian_packet(
    theta,
    center=np.pi,
    sigma=0.10
):
    """
    Pulso gaussiano inicial.
    """

    return np.exp(
        -(theta-center)**2
        /
        (2*sigma**2)
    )

Overwriting GER_CORE/ger_graph.py


In [2]:
import os

# 1. Cria a pasta física GER_CORE no painel esquerdo do Colab (se ela não existir)
pasta = "GER_CORE"
if not os.path.exists(pasta):
    os.makedirs(pasta)
    print(f"✅ Pasta '{pasta}' criada com sucesso no ambiente!")
else:
    print(f"ℹ️ A pasta '{pasta}' já existe no ambiente.")

# 2. Mostra onde ela está localizada
print("Diretório atual de trabalho:", os.getcwd())

✅ Pasta 'GER_CORE' criada com sucesso no ambiente!
Diretório atual de trabalho: /content


In [5]:
%%writefile GER_CORE/ger_potential.py

"""
=========================================================
GER CORE
Arquivo : ger_potential.py
=========================================================

Potenciais não lineares utilizados pela Geometria
Espectral Relacional.

Este módulo concentra toda a física não linear do
projeto.

Cada potencial retorna:

    força
    energia potencial

A interface pública é a classe:

    Potential.evaluate(...)
"""

from __future__ import annotations

import numpy as np


# =========================================================
# Potencial A
# =========================================================

def potential_A(gamma):
    """
    Potencial cúbico (φ⁴).

    V = γ⁴ / 4
    """

    force = gamma**3

    energy = -0.25 * gamma**4

    return force, energy


# =========================================================
# Potencial C
# =========================================================

def potential_C(gamma):
    """
    Potencial saturante.

    V = log(cosh(γ))
    """

    force = np.tanh(gamma)

    energy = np.log(np.cosh(gamma))

    return force, energy


# =========================================================
# Interface pública
# =========================================================

class Potential:
    """
    Interface única para todos os potenciais.

    Exemplo
    -------

    force, energy = Potential.evaluate(
        gamma,
        "A"
    )
    """

    @staticmethod
    def evaluate(gamma, potential="A"):

        potential = potential.upper()

        if potential == "A":

            return potential_A(gamma)

        elif potential == "C":

            return potential_C(gamma)

        raise ValueError(
            f"Potencial '{potential}' não reconhecido."
        )

Writing GER_CORE/ger_potential.py


In [7]:

%%writefile GER_CORE/ger_metrics.py
"""
=========================================================
GER CORE
Arquivo : ger_metrics.py
=========================================================

Métricas globais da Geometria Espectral Relacional.

Implementa:

• Energia Hamiltoniana
• Norma L²
• Amplitude máxima
• Erro relativo de energia
• Critério automático de divergência

GER CORE v1.0
=========================================================
"""

from __future__ import annotations

import numpy as np

from GER_CORE.ger_potential import Potential


# =========================================================
# Constantes
# =========================================================

ENERGY_TOL = 1e-4

DIVERGENCE_AMPLITUDE = 1e6

EPS = 1e-15


# =========================================================
# Norma L²
# =========================================================

def compute_l2_norm(gamma):
    """
    Norma L² do campo.
    """

    gamma = np.asarray(gamma)

    return np.sqrt(np.sum(gamma**2))


# =========================================================
# Amplitude máxima
# =========================================================

def compute_max_amplitude(gamma):
    """
    Máxima amplitude absoluta do campo.
    """

    gamma = np.asarray(gamma)

    return np.max(np.abs(gamma))


# =========================================================
# Energia Hamiltoniana
# =========================================================

def compute_hamiltonian(
    gamma,
    velocity,
    laplacian,
    beta,
    potential="A",
):
    """
    Calcula a energia Hamiltoniana discreta.

    H =
        T
      + E_elástica
      + E_não_linear

    Para comparações entre diferentes tamanhos
    de rede, utiliza-se a densidade média de
    energia.
    """

    gamma = np.asarray(gamma)
    velocity = np.asarray(velocity)

    kinetic = (
        0.5
        * np.mean(velocity**2)
    )

    elastic = (
        0.5
        * np.mean(
            gamma * (laplacian @ gamma)
        )
    )

    _, potential_energy = Potential.evaluate(
        gamma,
        potential,
    )

    nonlinear = (
        beta
        * np.mean(potential_energy)
    )

    return (
        kinetic
        + elastic
        + nonlinear
    )


# =========================================================
# Erro relativo
# =========================================================

def relative_energy_error(
    reference_energy,
    current_energy,
):
    """
    Erro relativo da energia.
    """

    return abs(
        current_energy - reference_energy
    ) / (
        abs(reference_energy) + EPS
    )


# =========================================================
# Divergência
# =========================================================

def check_divergence(
    gamma,
    energy_error,
):
    """
    Detecta explosões numéricas.
    """

    amplitude = compute_max_amplitude(gamma)

    if amplitude > DIVERGENCE_AMPLITUDE:
        return True

    if np.isnan(amplitude):
        return True

    if np.isinf(amplitude):
        return True

    if np.isnan(energy_error):
        return True

    if np.isinf(energy_error):
        return True

    return False

Writing GER_CORE/ger_metrics.py


In [8]:
%%writefile GER_CORE/ger_modal.py

"""
=========================================================
GER CORE
Arquivo : ger_modal.py
=========================================================

Observatório Espectral da Geometria Espectral Relacional.

Este módulo contém todas as ferramentas de análise
modal utilizadas pelas auditorias S26-B.

Responsabilidades:

• Projeção na base espectral
• Distribuição de energia modal
• Entropia espectral
• Centro e largura modal
• Participação modal
• Separação por bandas espectrais
"""

from __future__ import annotations

import numpy as np


EPS = 1e-15


# =========================================================
# Projeção modal
# =========================================================

def modal_projection(
    gamma,
    eigenvectors
):
    """
    Projeta o campo na base espectral.

    gamma_hat = V^T gamma
    """

    gamma = np.asarray(gamma)

    return eigenvectors.T @ gamma


# =========================================================
# Energia modal
# =========================================================

def modal_energy(
    gamma,
    eigenvectors
):
    """
    Energia associada a cada modo espectral.
    """

    coefficients = modal_projection(
        gamma,
        eigenvectors
    )

    energy = coefficients**2

    return energy


# =========================================================
# Probabilidade modal
# =========================================================

def modal_probability(
    gamma,
    eigenvectors
):
    """
    Normalização da energia modal.

    Soma das probabilidades = 1
    """

    energy = modal_energy(
        gamma,
        eigenvectors
    )

    total = np.sum(energy) + EPS

    return energy / total


# =========================================================
# Modo dominante
# =========================================================

def dominant_mode(probability):
    """
    Retorna o modo com maior concentração.
    """

    return int(
        np.argmax(probability)
    )


# =========================================================
# Centro espectral
# =========================================================

def spectral_center(probability):
    """
    Centro médio dos modos.
    """

    modes = np.arange(
        len(probability)
    )

    return np.sum(
        modes * probability
    )


# =========================================================
# Largura espectral
# =========================================================

def spectral_width(probability):
    """
    Desvio modal da distribuição espectral.
    """

    modes = np.arange(
        len(probability)
    )

    center = spectral_center(
        probability
    )

    variance = np.sum(
        probability *
        (modes - center)**2
    )

    return np.sqrt(
        variance
    )


# =========================================================
# Entropia espectral
# =========================================================

def spectral_entropy(probability):
    """
    Entropia de Shannon da distribuição modal.
    """

    p = probability[
        probability > 0
    ]

    return -np.sum(
        p * np.log(p)
    )


# =========================================================
# Participation Ratio
# =========================================================

def participation_ratio(probability):
    """
    Mede quantos modos participam efetivamente.

    PR = 1 / Σp²
    """

    return 1.0 / (
        np.sum(probability**2)
        + EPS
    )


# =========================================================
# Energia por bandas
# =========================================================

def spectral_bands(
    probability
):
    """
    Divide energia espectral em:

    baixa frequência
    média frequência
    alta frequência
    """

    n = len(probability)

    low_end = n // 3
    mid_end = 2 * n // 3

    low = np.sum(
        probability[:low_end]
    )

    medium = np.sum(
        probability[low_end:mid_end]
    )

    high = np.sum(
        probability[mid_end:]
    )

    return {
        "low": low,
        "medium": medium,
        "high": high
    }


# =========================================================
# Auditoria completa
# =========================================================

def analyze_modal_state(
    gamma,
    eigenvectors
):
    """
    Executa todo o observatório espectral.

    Retorna todas as métricas modais.
    """

    probability = modal_probability(
        gamma,
        eigenvectors
    )

    return {

        "modal_energy":
            modal_energy(
                gamma,
                eigenvectors
            ),

        "probability":
            probability,

        "dominant_mode":
            dominant_mode(
                probability
            ),

        "modal_center":
            spectral_center(
                probability
            ),

        "modal_width":
            spectral_width(
                probability
            ),

        "spectral_entropy":
            spectral_entropy(
                probability
            ),

        "participation_ratio":
            participation_ratio(
                probability
            ),

        "spectral_bands":
            spectral_bands(
                probability
            )
    }
# =========================================================
# Snapshot completo da simulação
# =========================================================

from GER_CORE.ger_metrics import (
    compute_hamiltonian,
    compute_l2_norm,
    compute_max_amplitude,
)


def build_snapshot(
    step,
    time,
    gamma,
    velocity,
    laplacian,
    beta,
    potential,
    eigenvectors,
    theta,
):
    """
    Constrói um snapshot completo do estado da simulação.

    Este formato é utilizado pelo GER CORE durante toda
    a evolução temporal.
    """

    modal = analyze_modal_state(
        gamma,
        eigenvectors,
    )

    snapshot = {

        "step": step,

        "time": time,

        "energy": compute_hamiltonian(
            gamma,
            velocity,
            laplacian,
            beta,
            potential,
        ),

        "l2": compute_l2_norm(gamma),

        "amplitude": compute_max_amplitude(gamma),

        "dominant_mode":
            modal["dominant_mode"],

        "modal_center":
            modal["modal_center"],

        "modal_width":
            modal["modal_width"],

        "spectral_entropy":
            modal["spectral_entropy"],

        "participation_ratio":
            modal["participation_ratio"],

        "probability":
            modal["probability"],

        "modal_energy":
            modal["modal_energy"],

        "spectral_bands":
            modal["spectral_bands"],
    }

    return snapshot

Writing GER_CORE/ger_modal.py


In [9]:
%%writefile GER_CORE/ger_snapshot.py

"""
=========================================================
GER CORE
Arquivo : ger_snapshot.py
=========================================================

Registro padronizado dos estados temporais da GER.

Cada snapshot representa uma fotografia completa
do sistema em um instante da evolução.

Responsabilidades:

• Métricas globais
• Observatório espectral
• Organização dos dados
"""

from __future__ import annotations


from GER_CORE.ger_metrics import (
    compute_hamiltonian,
    compute_l2_norm,
    compute_max_amplitude
)

from GER_CORE.ger_modal import (
    analyze_modal_state
)


# =========================================================
# Construção do Snapshot
# =========================================================

def build_snapshot(
    step,
    time,
    gamma,
    velocity,
    laplacian,
    beta,
    potential,
    eigenvectors,
    theta=None
):
    """
    Cria um registro completo do estado.

    Parameters
    ----------

    step :
        passo temporal

    time :
        tempo físico/numerico

    gamma :
        campo atual

    velocity :
        velocidade atual

    laplacian :
        operador discreto

    beta :
        intensidade não linear

    potential :
        tipo de potencial

    eigenvectors :
        base espectral
    """


    # -----------------------------
    # Métricas globais
    # -----------------------------

    energy = compute_hamiltonian(
        gamma,
        velocity,
        laplacian,
        beta,
        potential
    )

    l2 = compute_l2_norm(
        gamma
    )

    amplitude = compute_max_amplitude(
        gamma
    )


    # -----------------------------
    # Análise espectral
    # -----------------------------

    modal = analyze_modal_state(
        gamma,
        eigenvectors
    )


    # -----------------------------
    # Registro final
    # -----------------------------

    snapshot = {

        "step":
            step,

        "time":
            time,

        "energy":
            energy,

        "l2":
            l2,

        "amplitude":
            amplitude,

        "dominant_mode":
            modal["dominant_mode"],

        "modal_center":
            modal["modal_center"],

        "modal_width":
            modal["modal_width"],

        "spectral_entropy":
            modal["spectral_entropy"],

        "participation_ratio":
            modal["participation_ratio"],

        "probability":
            modal["probability"],

        "modal_energy":
            modal["modal_energy"],

        "spectral_bands":
            modal["spectral_bands"],

        "gamma":
            gamma.copy()
    }


    return snapshot

Writing GER_CORE/ger_snapshot.py


In [10]:
%%writefile GER_CORE/ger_engine.py
"""
=========================================================
GER CORE
Arquivo : ger_engine.py
=========================================================

Motor numérico principal da Geometria Espectral Relacional.

Responsabilidades
-----------------

• Construção da geometria
• Inicialização do integrador Verlet
• Evolução temporal
• Auditoria automática
• Geração dos snapshots
• Retorno completo da simulação

Versão
-------

GER CORE v1.0
"""

from __future__ import annotations

import numpy as np

from GER_CORE.ger_graph import (
    build_ring_graph,
    spectral_basis,
    gaussian_packet,
)

from GER_CORE.ger_metrics import (
    compute_hamiltonian,
    relative_energy_error,
    check_divergence,
)

from GER_CORE.ger_snapshot import (
    build_snapshot,
)

from GER_CORE.ger_potential import (
    Potential,
)


# =========================================================
# Inicialização do integrador de Verlet
# =========================================================

def initialize_verlet(
    gamma,
    laplacian,
    beta,
    potential,
    dt,
):
    """
    Constrói o estado anterior necessário pelo
    esquema de Verlet.

    Parameters
    ----------
    gamma : ndarray
        Campo inicial.

    laplacian : ndarray
        Laplaciano da rede.

    beta : float
        Intensidade da não linearidade.

    potential : str
        Identificador do potencial.

    dt : float
        Passo temporal.
    """

    force, _ = Potential.evaluate(
        gamma,
        potential,
    )

    acceleration = (
        -(laplacian @ gamma)
        + beta * force
    )

    gamma_old = (
        gamma
        - 0.5 * dt**2 * acceleration
    )

    return gamma_old


# =========================================================
# Velocidade central
# =========================================================

def central_velocity(
    gamma_new,
    gamma_old,
    dt,
):
    """
    Aproximação centrada da velocidade.
    """

    return (
        gamma_new - gamma_old
    ) / (2.0 * dt)
    # =========================================================
# Motor principal
# =========================================================

def run_engine(
    n=384,
    timesteps=2000,
    dt=2.5e-4,
    beta=1.0,
    potential="A",
    snapshot_stride=50,
    sigma=0.10,
):
    """
    Executa uma simulação completa da Geometria
    Espectral Relacional.

    Retorna um dicionário contendo todos os dados
    necessários para auditorias posteriores.
    """

    # -----------------------------------------------------
    # Construção da geometria
    # -----------------------------------------------------

    _, laplacian, theta = build_ring_graph(n)

    eigenvalues, eigenvectors = spectral_basis(
        laplacian
    )

    # -----------------------------------------------------
    # Condição inicial
    # -----------------------------------------------------

    gamma = gaussian_packet(
        theta,
        center=np.pi,
        sigma=sigma,
    )

    gamma_old = initialize_verlet(
        gamma=gamma,
        laplacian=laplacian,
        beta=beta,
        potential=potential,
        dt=dt,
    )

    velocity = central_velocity(
        gamma,
        gamma_old,
        dt,
    )

    energy_initial = compute_hamiltonian(
        gamma=gamma,
        velocity=velocity,
        laplacian=laplacian,
        beta=beta,
        potential=potential,
    )

    snapshots = []

    diverged = False

    # -----------------------------------------------------
    # Evolução temporal
    # -----------------------------------------------------

    for step in range(timesteps):

        force, _ = Potential.evaluate(
            gamma,
            potential,
        )

        acceleration = (
            -(laplacian @ gamma)
            + beta * force
        )

        gamma_new = (
            2.0 * gamma
            - gamma_old
            + dt**2 * acceleration
        )

        velocity = central_velocity(
            gamma_new,
            gamma_old,
            dt,
        )
                # -------------------------------------------------
        # Auditoria energética
        # -------------------------------------------------

        energy = compute_hamiltonian(
            gamma=gamma_new,
            velocity=velocity,
            laplacian=laplacian,
            beta=beta,
            potential=potential,
        )

        energy_error = relative_energy_error(
            energy_initial,
            energy,
        )

        if check_divergence(
            gamma_new,
            energy_error,
        ):
            diverged = True

        # -------------------------------------------------
        # Snapshot
        # -------------------------------------------------

        if (
            step % snapshot_stride == 0
            or step == timesteps - 1
        ):

            snapshot = build_snapshot(
                step=step,
                time=step * dt,
                gamma=gamma_new,
                velocity=velocity,
                laplacian=laplacian,
                beta=beta,
                potential=potential,
                eigenvectors=eigenvectors,
                theta=theta,
            )

            snapshot["energy_error"] = energy_error

            snapshots.append(snapshot)

        # -------------------------------------------------
        # Atualização temporal
        # -------------------------------------------------

        gamma_old = gamma
        gamma = gamma_new
            # -----------------------------------------------------
    # Estado final
    # -----------------------------------------------------

    final_velocity = central_velocity(
        gamma,
        gamma_old,
        dt,
    )

    final_energy = compute_hamiltonian(
        gamma=gamma,
        velocity=final_velocity,
        laplacian=laplacian,
        beta=beta,
        potential=potential,
    )

    final_error = relative_energy_error(
        energy_initial,
        final_energy,
    )

    # -----------------------------------------------------
    # Retorno
    # -----------------------------------------------------

    return {

        "configuration": {

            "n": n,
            "dt": dt,
            "timesteps": timesteps,
            "beta": beta,
            "potential": potential,
            "sigma": sigma,

        },

        "initial": {

            "energy": energy_initial,

        },

        "final": {

            "energy": final_energy,
            "energy_error": final_error,

        },

        "snapshots": snapshots,

        "gamma": gamma,

        "laplacian": laplacian,

        "eigenvalues": eigenvalues,

        "eigenvectors": eigenvectors,

        "diverged": diverged,

        }

Writing GER_CORE/ger_engine.py


In [11]:
%%writefile GER_CORE/ger_validation.py

"""
=========================================================
GER CORE
Arquivo : ger_validation.py
=========================================================

Validação automática do núcleo computacional GER.

Executa testes mínimos para garantir que:

• módulos carregam corretamente
• geometria funciona
• motor executa
• snapshots são produzidos
• observatório espectral está ativo
"""

from __future__ import annotations


import numpy as np


from GER_CORE.ger_graph import (
    build_ring_graph,
    spectral_basis,
    gaussian_packet
)


from GER_CORE.ger_engine import (
    run_engine
)



# =========================================================
# Validação da geometria
# =========================================================

def validate_graph():

    A, L, theta = build_ring_graph(
        64
    )

    assert A.shape == (
        64,
        64
    )

    assert L.shape == (
        64,
        64
    )

    assert len(theta) == 64


    return True



# =========================================================
# Validação espectral
# =========================================================

def validate_spectrum():

    _, L, _ = build_ring_graph(
        64
    )


    eigenvalues, eigenvectors = (
        spectral_basis(L)
    )


    assert len(eigenvalues) == 64

    assert eigenvectors.shape == (
        64,
        64
    )


    return True



# =========================================================
# Validação do motor
# =========================================================

def validate_engine():

    result = run_engine(
        n=128,
        timesteps=100,
        dt=2.5e-4,
        beta=1.0,
        potential="A",
        snapshot_stride=20
    )


    assert "snapshots" in result

    assert len(
        result["snapshots"]
    ) > 0


    assert (
        result["diverged"]
        is False
    )


    snapshot = (
        result["snapshots"][0]
    )


    required = [

        "energy",

        "dominant_mode",

        "modal_width",

        "spectral_entropy"

    ]


    for key in required:

        assert key in snapshot


    return True



# =========================================================
# Validação completa
# =========================================================

def validate_GER_CORE():

    print(
        "================================"
    )

    print(
        " GER CORE VALIDATION"
    )

    print(
        "================================"
    )


    tests = [

        (
            "Geometria",
            validate_graph
        ),

        (
            "Espectro",
            validate_spectrum
        ),

        (
            "Motor",
            validate_engine
        )

    ]


    for name, test in tests:

        test()

        print(
            f"[OK] {name}"
        )


    print(
        "================================"
    )

    print(
        " GER CORE VALIDADO "
    )

    print(
        "================================"
    )


    return True

Writing GER_CORE/ger_validation.py


In [12]:
%%writefile GER_CORE/bootstrap.py

"""
=========================================================
GER CORE
Arquivo : bootstrap.py
=========================================================

Inicializador oficial da Geometria Espectral Relacional.

Uso:

    python bootstrap.py

ou no Google Colab:

    %run GER_CORE/bootstrap.py


Carrega:

- geometria
- potenciais
- métricas
- análise modal
- snapshots
- motor temporal

e executa validação automática.
"""



# =========================================================
# Imports principais
# =========================================================

from GER_CORE.ger_engine import (
    run_engine
)


from GER_CORE.ger_validation import (
    validate_GER_CORE
)



# =========================================================
# Inicialização
# =========================================================

def initialize():

    print(
        "================================"
    )

    print(
        " INICIANDO GER CORE v1"
    )

    print(
        "================================"
    )


    validate_GER_CORE()


    print(
        ""
    )

    print(
        "GER CORE pronto para experimentos."
    )

    print(
        "================================"
    )


    return True



# =========================================================
# Execução direta
# =========================================================

if __name__ == "__main__":

    initialize()

Writing GER_CORE/bootstrap.py


In [13]:

%run GER_CORE/bootstrap.py

 INICIANDO GER CORE v1
 GER CORE VALIDATION
[OK] Geometria
[OK] Espectro
[OK] Motor
 GER CORE VALIDADO 

GER CORE pronto para experimentos.


In [14]:
%%writefile GER_CORE/__init__.py

"""
GER CORE

Biblioteca computacional da
Geometria Espectral Relacional.
"""

Writing GER_CORE/__init__.py


In [15]:
resultado_n192 = run_engine(
    n=192,
    timesteps=4000,
    dt=1.25e-4,
    beta=1.0,
    potential="A"
)

resultado_n384 = run_engine(
    n=384,
    timesteps=4000,
    dt=1.25e-4,
    beta=1.0,
    potential="A"
)

resultado_n768 = run_engine(
    n=768,
    timesteps=4000,
    dt=1.25e-4,
    beta=1.0,
    potential="A"
)


print("n=192:", resultado_n192["final"])
print("n=384:", resultado_n384["final"])
print("n=768:", resultado_n768["final"])

n=192: {'energy': np.float64(-0.005791695481486014), 'energy_error': np.float64(0.36546795462502774)}
n=384: {'energy': np.float64(-0.006617031433447576), 'energy_error': np.float64(0.3789516241006723)}
n=768: {'energy': np.float64(-0.006832377076743902), 'energy_error': np.float64(0.3831801744893025)}


In [16]:
import inspect
from GER_CORE.ger_metrics import compute_hamiltonian

print(inspect.getsource(compute_hamiltonian))

def compute_hamiltonian(
    gamma,
    velocity,
    laplacian,
    beta,
    potential="A",
):
    """
    Calcula a energia Hamiltoniana discreta.

    H =
        T
      + E_elástica
      + E_não_linear

    Para comparações entre diferentes tamanhos
    de rede, utiliza-se a densidade média de
    energia.
    """

    gamma = np.asarray(gamma)
    velocity = np.asarray(velocity)

    kinetic = (
        0.5
        * np.mean(velocity**2)
    )

    elastic = (
        0.5
        * np.mean(
            gamma * (laplacian @ gamma)
        )
    )

    _, potential_energy = Potential.evaluate(
        gamma,
        potential,
    )

    nonlinear = (
        beta
        * np.mean(potential_energy)
    )

    return (
        kinetic
        + elastic
        + nonlinear
    )



In [17]:
print("Energia por nó:")
print("n192:", resultado_n192["final"]["energy"]/192)
print("n384:", resultado_n384["final"]["energy"]/384)
print("n768:", resultado_n768["final"]["energy"]/768)

Energia por nó:
n192: -3.0165080632739655e-05
n384: -1.723185269126973e-05
n768: -8.896324318676955e-06


In [18]:

%%writefile GER_CORE/ger_convergence.py
"""
=========================================================
GER CORE
Arquivo : ger_convergence.py
=========================================================

Rotinas de validação de convergência da Geometria
Espectral Relacional.

Este módulo implementa:

• Densidade de energia
• Comparação temporal
• Comparação espacial
• Auditoria de estabilidade

Objetivo:
Garantir que os resultados numéricos sejam robustos
antes das análises espectrais da Série S26-B.

=========================================================
"""

from __future__ import annotations

import numpy as np


# =========================================================
# Densidade de energia
# =========================================================

def energy_density(energy, n):
    """
    Calcula energia por grau de liberdade.

    ε = H / N

    Parâmetros
    ----------
    energy : float
        Energia Hamiltoniana total.

    n : int
        Número de nós da rede.

    Retorno
    -------
    float
    """

    return energy / n



# =========================================================
# Erro relativo entre resoluções
# =========================================================

def relative_difference(a, b):
    """
    Diferença relativa entre dois valores.
    """

    return abs(a - b) / (
        abs(b) + 1e-15
    )



# =========================================================
# Auditoria espacial
# =========================================================

def spatial_convergence_test(results):
    """
    Analisa convergência espacial.

    Espera um dicionário:

    {
        n : resultado_run_engine
    }

    Exemplo:

    {
        192: result192,
        384: result384,
        768: result768
    }

    """

    report = []


    for n, result in sorted(results.items()):

        energy = result["final"]["energy"]

        error = result["final"]["energy_error"]

        density = energy_density(
            energy,
            n
        )

        report.append(
            {
                "n": n,
                "energy": energy,
                "energy_density": density,
                "energy_error": error
            }
        )


    return report



# =========================================================
# Auditoria temporal
# =========================================================

def temporal_convergence_test(results):
    """
    Analisa convergência temporal.

    Espera:

    {
        dt : resultado_run_engine
    }

    """

    report = []


    for dt, result in sorted(
        results.items(),
        reverse=True
    ):

        report.append(
            {
                "dt": dt,
                "energy":
                    result["final"]["energy"],

                "energy_error":
                    result["final"]["energy_error"]
            }
        )


    return report



# =========================================================
# Impressão padronizada
# =========================================================

def print_convergence_report(report):
    """
    Exibe tabela simples de convergência.
    """

    for item in report:

        print(
            item
        )

Writing GER_CORE/ger_convergence.py


In [19]:
from GER_CORE.ger_convergence import (
    spatial_convergence_test,
    print_convergence_report
)

In [20]:
report = spatial_convergence_test(
    {
        192: resultado_n192,
        384: resultado_n384,
        768: resultado_n768
    }
)

print_convergence_report(report)

{'n': 192, 'energy': np.float64(-0.005791695481486014), 'energy_density': np.float64(-3.0165080632739655e-05), 'energy_error': np.float64(0.36546795462502774)}
{'n': 384, 'energy': np.float64(-0.006617031433447576), 'energy_density': np.float64(-1.723185269126973e-05), 'energy_error': np.float64(0.3789516241006723)}
{'n': 768, 'energy': np.float64(-0.006832377076743902), 'energy_density': np.float64(-8.896324318676955e-06), 'energy_error': np.float64(0.3831801744893025)}


In [21]:

%%writefile GER_CORE/ger_reversibility.py

"""
=========================================================
GER CORE
Arquivo : ger_reversibility.py
=========================================================

Auditoria de reversibilidade temporal.

Esta primeira versão prepara a infraestrutura para os
testes de reversibilidade do integrador Verlet.

=========================================================
"""

from __future__ import annotations

import numpy as np


# =========================================================
# Norma relativa
# =========================================================

def relative_norm(a, b):
    """
    Erro relativo entre dois vetores.
    """

    a = np.asarray(a)
    b = np.asarray(b)

    den = np.linalg.norm(a)

    if den < 1e-15:
        den = 1.0

    return np.linalg.norm(a - b) / den


# =========================================================
# Comparação de estados
# =========================================================

def compare_states(reference, recovered):
    """
    Compara dois estados do sistema.
    """

    return {

        "gamma_error":
            relative_norm(
                reference["gamma"],
                recovered["gamma"]
            ),

        "energy_error":
            abs(
                reference["final"]["energy"]
                -
                recovered["final"]["energy"]
            ),

        "l2_error":
            abs(
                reference["snapshots"][-1]["l2"]
                -
                recovered["snapshots"][-1]["l2"]
            ),

        "amplitude_error":
            abs(
                reference["snapshots"][-1]["amplitude"]
                -
                recovered["snapshots"][-1]["amplitude"]
            )
    }


# =========================================================
# Relatório
# =========================================================

def print_reversibility_report(report):

    print("=" * 60)
    print("AUDITORIA DE REVERSIBILIDADE")
    print("=" * 60)

    for key, value in report.items():

        print(f"{key}: {value}")

Writing GER_CORE/ger_reversibility.py


In [22]:
from GER_CORE.bootstrap import *

In [23]:
from GER_CORE.bootstrap import *

In [24]:
from GER_CORE.bootstrap import initialize

initialize()

 INICIANDO GER CORE v1
 GER CORE VALIDATION
[OK] Geometria
[OK] Espectro
[OK] Motor
 GER CORE VALIDADO 

GER CORE pronto para experimentos.


True

In [25]:
resultado = run_engine()

print(resultado["final"])

{'energy': np.float64(-0.006618340104149309), 'energy_error': np.float64(0.3792243501415257)}


In [26]:
from GER_CORE.ger_engine import run_engine


for dt in [1e-3, 5e-4, 2.5e-4, 1e-4, 5e-5]:

    result = run_engine(
        n=384,
        timesteps=2000,
        dt=dt,
        beta=1.0,
        potential="A"
    )

    print(
        "dt:",
        dt,
        "| Energia:",
        result["final"]["energy"],
        "| Erro:",
        result["final"]["energy_error"]
    )

/content/GER_CORE/ger_potential.py:42: RuntimeWarning: overflow encountered in power
  energy = -0.25 * gamma**4
/content/GER_CORE/ger_metrics.py:96: RuntimeWarning: overflow encountered in square
  * np.mean(velocity**2)
/content/GER_CORE/ger_metrics.py:102: RuntimeWarning: overflow encountered in multiply
  gamma * (laplacian @ gamma)
/content/GER_CORE/ger_potential.py:40: RuntimeWarning: overflow encountered in power
  force = gamma**3
/content/GER_CORE/ger_metrics.py:117: RuntimeWarning: invalid value encountered in scalar add
  kinetic
/content/GER_CORE/ger_metrics.py:102: RuntimeWarning: invalid value encountered in matmul
  gamma * (laplacian @ gamma)
/content/GER_CORE/ger_engine.py:196: RuntimeWarning: invalid value encountered in matmul
  -(laplacian @ gamma)
/content/GER_CORE/ger_engine.py:196: RuntimeWarning: invalid value encountered in add
  -(laplacian @ gamma)


dt: 0.001 | Energia: nan | Erro: nan
dt: 0.0005 | Energia: -0.021413534844257738 | Erro: 3.4624586790921863
dt: 0.00025 | Energia: -0.006618340104149309 | Erro: 0.3792243501415257
dt: 0.0001 | Energia: -0.005039570199095989 | Erro: 0.05021769607121459
dt: 5e-05 | Energia: -0.004857305095520187 | Erro: 0.01223468691520569


In [27]:
import GER_CORE.ger_engine as ge

print(
    [
        nome for nome in dir(ge)
        if not nome.startswith("_")
    ]
)

['Potential', 'annotations', 'build_ring_graph', 'build_snapshot', 'central_velocity', 'check_divergence', 'compute_hamiltonian', 'gaussian_packet', 'initialize_verlet', 'np', 'relative_energy_error', 'run_engine', 'spectral_basis']


In [28]:
import numpy as np

from GER_CORE.ger_graph import (
    build_ring_graph,
    spectral_basis,
    gaussian_packet,
)

from GER_CORE.ger_engine import (
    initialize_verlet,
    central_velocity,
)

from GER_CORE.ger_potential import Potential


def forward_step(
    gamma,
    gamma_old,
    L,
    beta,
    dt,
    potential="A"
):

    force, _ = Potential.evaluate(
        gamma,
        potential
    )

    acceleration = (
        -(L @ gamma)
        +
        beta * force
    )

    gamma_new = (
        2.0 * gamma
        -
        gamma_old
        +
        dt**2 * acceleration
    )

    return gamma_new


# parâmetros

n = 384
dt = 5e-5
steps = 500

_, L, theta = build_ring_graph(n)

gamma0 = gaussian_packet(theta)

gamma_old = initialize_verlet(
    gamma0,
    L,
    1.0,
    "A",
    dt
)

gamma = gamma0.copy()


# evolução para frente

for _ in range(steps):

    gamma_new = forward_step(
        gamma,
        gamma_old,
        L,
        1.0,
        dt,
    )

    gamma_old = gamma
    gamma = gamma_new


gamma_forward = gamma.copy()


# reversão aproximada

gamma_reverse = gamma_forward.copy()
gamma_reverse_old = gamma_old.copy()


for _ in range(steps):

    gamma_new = forward_step(
        gamma_reverse,
        gamma_reverse_old,
        L,
        1.0,
        -dt,
    )

    gamma_reverse_old = gamma_reverse
    gamma_reverse = gamma_new


error = np.linalg.norm(
    gamma_reverse - gamma0
) / (
    np.linalg.norm(gamma0)
    + 1e-15
)


print("Erro de reversibilidade:", error)

Erro de reversibilidade: 0.0009290588402990414


In [29]:
import numpy as np
from GER_CORE.ger_potential import potential_A


gamma = np.array([
    0.1,
    0.5,
    1.0
])


force, energy = potential_A(gamma)


epsilon = 1e-6

V_plus = -0.25 * (gamma + epsilon)**4
V_minus = -0.25 * (gamma - epsilon)**4


numerical_force = -(
    V_plus - V_minus
) / (2 * epsilon)


print("Força código:", force)
print("Força derivada:", numerical_force)
print("Diferença:", force - numerical_force)

Força código: [0.001 0.125 1.   ]
Força derivada: [0.001 0.125 1.   ]
Diferença: [-1.00448296e-13 -1.85973459e-12  1.28776989e-11]


In [28]:
from GER_CORE.ger_engine import run_engine


for n in [96, 192, 384, 768]:

    result = run_engine(
        n=n,
        timesteps=2000,
        dt=5e-5,
        beta=1.0,
        potential="A"
    )

    final = result["final"]

    gamma = result["gamma"]

    print(
        "\nN =", n
    )

    print(
        "Energia:",
        final["energy"]
    )

    print(
        "Energia/n:",
        final["energy"] / n
    )

    print(
        "Erro:",
        final["energy_error"]
    )

    print(
        "Amplitude:",
        max(abs(gamma))
    )


N = 96
Energia: -0.0021511741697794666
Energia/n: -2.240806426853611e-05
Erro: 0.013745450609571098
Amplitude: 1.0030804544267478

N = 192
Energia: -0.004292180742146636
Energia/n: -2.235510803201373e-05
Erro: 0.011937742585628826
Amplitude: 1.004493241915854

N = 384
Energia: -0.004857305095520187
Energia/n: -1.2649232019583822e-05
Erro: 0.01223468691520569
Amplitude: 1.0048838011936252

N = 768
Energia: -0.0050005384873278765
Energia/n: -6.511117822041506e-06
Erro: 0.012333717901273075
Amplitude: 1.0049839518090575


In [30]:
from GER_CORE.ger_engine import run_engine


for potential in ["A", "C"]:

    result = run_engine(
        n=384,
        timesteps=2000,
        dt=5e-5,
        beta=1.0,
        potential=potential
    )

    print("\nPotencial:", potential)

    print(
        "Energia:",
        result["final"]["energy"]
    )

    print(
        "Erro:",
        result["final"]["energy_error"]
    )

    print(
        "Amplitude:",
        max(abs(result["gamma"]))
    )


Potencial: A
Energia: -0.004857305095520187
Erro: 0.01223468691520569
Amplitude: 1.0048838011936252

Potencial: C
Energia: 0.013037804433495876
Erro: 0.009228854240529626
Amplitude: 1.0036799218677608


In [31]:
from GER_CORE.ger_engine import run_engine
import numpy as np


result = run_engine(
    n=384,
    timesteps=2000,
    dt=5e-5,
    beta=1.0,
    potential="A",
    snapshot_stride=100
)


for s in result["snapshots"]:

    print(
        "step:",
        s["step"],
        "| modo:",
        s["dominant_mode"],
        "| entropia:",
        s["spectral_entropy"],
        "| participação:",
        s["participation_ratio"],
        "| largura:",
        s["modal_width"]
    )

step: 0 | modo: 1 | entropia: 2.7209094984644855 | participação: 13.053917340768423 | largura: 8.653677637155312
step: 100 | modo: 1 | entropia: 2.720913630583644 | participação: 13.053962128063068 | largura: 8.65372080192743
step: 200 | modo: 1 | entropia: 2.7209257093125077 | participação: 13.054093047205818 | largura: 8.653846980981548
step: 300 | modo: 1 | entropia: 2.7209457353175637 | participação: 13.054310105359804 | largura: 8.65405618933073
step: 400 | modo: 1 | entropia: 2.720973709702843 | participação: 13.054613314401411 | largura: 8.65434845186148
step: 500 | modo: 1 | entropia: 2.7210096340086922 | participação: 13.05500269092125 | largura: 8.65472380332849
step: 600 | modo: 1 | entropia: 2.7210535102101607 | participação: 13.055478256225443 | largura: 8.655182288347211
step: 700 | modo: 1 | entropia: 2.721105340715107 | participação: 13.056040036337421 | largura: 8.655723961384597
step: 800 | modo: 1 | entropia: 2.7211651283620104 | participação: 13.056688062000333 | la

In [32]:
from GER_CORE.ger_engine import run_engine


for sigma in [0.05, 0.10, 0.20]:

    result = run_engine(
        n=384,
        timesteps=2000,
        dt=5e-5,
        beta=1.0,
        potential="A",
        snapshot_stride=200,
        sigma=sigma
    )

    final = result["snapshots"][-1]

    print("\n======================")
    print("Sigma:", sigma)
    print("======================")

    print(
        "Energia:",
        result["final"]["energy"]
    )

    print(
        "Erro:",
        result["final"]["energy_error"]
    )

    print(
        "Amplitude:",
        max(abs(result["gamma"]))
    )

    print(
        "Modo dominante:",
        final["dominant_mode"]
    )

    print(
        "Entropia:",
        final["spectral_entropy"]
    )

    print(
        "Participação:",
        final["participation_ratio"]
    )

    print(
        "Largura modal:",
        final["modal_width"]
    )


Sigma: 0.05
Energia: -0.0021460903710733225
Erro: 0.011937742585625988
Amplitude: 1.004493241915854
Modo dominante: 1
Entropia: 3.3957391079855763
Participação: 25.60100608089999
Largura modal: 17.170170852501077

Sigma: 0.1
Energia: -0.004857305095520187
Erro: 0.01223468691520569
Amplitude: 1.0048838011936252
Modo dominante: 1
Entropia: 2.7225032495388497
Participação: 13.071191868299676
Largura modal: 8.670361691053232

Sigma: 0.2
Energia: -0.01000107697465575
Erro: 0.012333717901274506
Amplitude: 1.0049839518090575
Modo dominante: 1
Entropia: 2.0685089221945323
Participação: 6.819258804099877
Largura modal: 4.297850358443465


In [33]:
from GER_CORE.ger_engine import run_engine
from GER_CORE.ger_reversibility import relative_norm


for dt in [1e-4, 5e-5, 2.5e-5]:

    result_forward = run_engine(
        n=384,
        timesteps=2000,
        dt=dt,
        beta=1.0,
        potential="A",
        sigma=0.10
    )

    gamma_final = result_forward["gamma"]


    # volta a partir do estado final
    result_backward = run_engine(
        n=384,
        timesteps=2000,
        dt=dt,
        beta=1.0,
        potential="A",
        sigma=0.10
    )

    gamma_return = result_backward["gamma"]


    error = relative_norm(
        gamma_final,
        gamma_return
    )


    print(
        "dt:",
        dt,
        "| erro reversibilidade:",
        error
    )

dt: 0.0001 | erro reversibilidade: 0.0
dt: 5e-05 | erro reversibilidade: 0.0
dt: 2.5e-05 | erro reversibilidade: 0.0


In [34]:

%%writefile GER_CORE/ger_reversibility_test.py

"""
=========================================================
GER CORE
Arquivo : ger_reversibility_test.py
=========================================================

Auditoria rigorosa de reversibilidade temporal.

Método:

1. Evolui o sistema para frente.
2. Captura estado final.
3. Inverte a velocidade.
4. Evolui novamente.
5. Compara com o estado inicial.

Não modifica o motor.
Apenas audita o integrador.
=========================================================
"""

from __future__ import annotations

import numpy as np


from GER_CORE.ger_graph import (
    build_ring_graph,
    spectral_basis,
    gaussian_packet
)


from GER_CORE.ger_engine import (
    initialize_verlet
)


from GER_CORE.ger_potential import (
    Potential
)



# =========================================================
# Norma relativa
# =========================================================

def relative_error(a, b):

    return (
        np.linalg.norm(a - b)
        /
        (np.linalg.norm(a) + 1e-15)
    )



# =========================================================
# Evolução simples Verlet reversível
# =========================================================

def verlet_step(
    gamma,
    gamma_old,
    L,
    beta,
    potential,
    dt
):

    force, _ = Potential.evaluate(
        gamma,
        potential
    )

    acceleration = (
        -(L @ gamma)
        +
        beta * force
    )

    gamma_new = (
        2.0 * gamma
        -
        gamma_old
        +
        dt**2 * acceleration
    )

    return gamma_new



# =========================================================
# Teste principal
# =========================================================

def test_reversibility(
    n=384,
    timesteps=2000,
    dt=5e-5,
    beta=1.0,
    potential="A",
    sigma=0.10
):


    _, L, theta = build_ring_graph(n)


    gamma_initial = gaussian_packet(
        theta,
        sigma=sigma
    )


    gamma = gamma_initial.copy()


    gamma_old = initialize_verlet(
        gamma,
        L,
        beta,
        potential,
        dt
    )


    # -------------------------
    # Evolução para frente
    # -------------------------

    for _ in range(timesteps):

        gamma_new = verlet_step(
            gamma,
            gamma_old,
            L,
            beta,
            potential,
            dt
        )

        gamma_old = gamma
        gamma = gamma_new



    gamma_final = gamma.copy()



    # velocidade aproximada final

    velocity_final = (
        gamma
        -
        gamma_old
    ) / dt



    # reconstrução reversa

    gamma_reverse_old = (
        gamma_final
        +
        dt * velocity_final
    )


    gamma_reverse = gamma_final.copy()



    # -------------------------
    # Evolução reversa
    # -------------------------

    for _ in range(timesteps):

        gamma_new = verlet_step(
            gamma_reverse,
            gamma_reverse_old,
            L,
            beta,
            potential,
            dt
        )

        gamma_reverse_old = gamma_reverse
        gamma_reverse = gamma_new



    error = relative_error(
        gamma_reverse,
        gamma_initial
    )


    return error

Writing GER_CORE/ger_reversibility_test.py


In [35]:
from GER_CORE.ger_reversibility_test import test_reversibility


for dt in [1e-4, 5e-5, 2.5e-5]:

    error = test_reversibility(
        dt=dt
    )

    print(
        "dt:",
        dt,
        "| erro reversibilidade:",
        error
    )

dt: 0.0001 | erro reversibilidade: 1.5876454832742025e-05
dt: 5e-05 | erro reversibilidade: 3.770554136815366e-06
dt: 2.5e-05 | erro reversibilidade: 9.306640253603175e-07


In [36]:
%%writefile GER_CORE/ger_symmetry_test.py

"""
=========================================================
GER CORE
Arquivo : ger_symmetry_test.py
=========================================================

Auditoria de simetria dinâmica do anel.

Testa se uma rotação da condição inicial
gera uma trajetória equivalente, também rotacionada.

=========================================================
"""

from __future__ import annotations

import numpy as np


from GER_CORE.ger_graph import (
    build_ring_graph,
    gaussian_packet
)

from GER_CORE.ger_potential import (
    Potential
)

from GER_CORE.ger_engine import (
    initialize_verlet
)


# =========================================================
# Passo Verlet
# =========================================================

def verlet_step(
    gamma,
    gamma_old,
    L,
    beta,
    potential,
    dt
):

    force, _ = Potential.evaluate(
        gamma,
        potential
    )

    acceleration = (
        -(L @ gamma)
        +
        beta * force
    )

    gamma_new = (
        2.0 * gamma
        -
        gamma_old
        +
        dt**2 * acceleration
    )

    return gamma_new



# =========================================================
# Evolução auxiliar
# =========================================================

def evolve(
    gamma,
    L,
    beta,
    potential,
    dt,
    timesteps
):

    gamma_old = initialize_verlet(
        gamma,
        L,
        beta,
        potential,
        dt
    )


    for _ in range(timesteps):

        gamma_new = verlet_step(
            gamma,
            gamma_old,
            L,
            beta,
            potential,
            dt
        )

        gamma_old = gamma
        gamma = gamma_new


    return gamma



# =========================================================
# Teste principal
# =========================================================

def test_dynamic_symmetry(
    n=384,
    shift=50,
    dt=5e-5,
    timesteps=2000
):

    _, L, theta = build_ring_graph(n)


    gamma = gaussian_packet(
        theta,
        sigma=0.10
    )


    gamma_shifted = np.roll(
        gamma,
        shift
    )


    final_original = evolve(
        gamma.copy(),
        L,
        1.0,
        "A",
        dt,
        timesteps
    )


    final_shifted = evolve(
        gamma_shifted.copy(),
        L,
        1.0,
        "A",
        dt,
        timesteps
    )


    # desfaz rotação

    final_shifted_back = np.roll(
        final_shifted,
        -shift
    )


    field_difference = (
        np.linalg.norm(
            final_original - final_shifted_back
        )
        /
        (
            np.linalg.norm(final_original)
            +1e-15
        )
    )


    energy_original = np.linalg.norm(
        final_original
    )

    energy_shifted = np.linalg.norm(
        final_shifted_back
    )


    amplitude_original = np.max(
        abs(final_original)
    )

    amplitude_shifted = np.max(
        abs(final_shifted_back)
    )


    return {

        "field_difference":
            field_difference,

        "norm_difference":
            abs(
                energy_original-energy_shifted
            )
            /
            (
                abs(energy_original)+1e-15
            ),

        "amplitude_difference":
            abs(
                amplitude_original-amplitude_shifted
            )
            /
            (
                abs(amplitude_original)+1e-15
            )
    }

Writing GER_CORE/ger_symmetry_test.py


In [37]:
from GER_CORE.ger_symmetry_test import test_ring_symmetry


result = test_ring_symmetry()


print(
    "Diferença após desfazer rotação:",
    result["initial_difference"]
)

ImportError: cannot import name 'test_ring_symmetry' from 'GER_CORE.ger_symmetry_test' (/content/GER_CORE/ger_symmetry_test.py)

In [38]:
from GER_CORE.ger_symmetry_test import test_dynamic_symmetry


result = test_dynamic_symmetry()


print(result)

{'field_difference': np.float64(0.0), 'norm_difference': np.float64(0.0), 'amplitude_difference': np.float64(0.0)}


In [39]:
!grep "test_dynamic_symmetry" GER_CORE/ger_symmetry_test.py

def test_dynamic_symmetry(


In [40]:
import importlib
import GER_CORE.ger_symmetry_test as gst

importlib.reload(gst)

<module 'GER_CORE.ger_symmetry_test' from '/content/GER_CORE/ger_symmetry_test.py'>

In [41]:
result = gst.test_dynamic_symmetry()

print(result)

{'field_difference': np.float64(0.0), 'norm_difference': np.float64(0.0), 'amplitude_difference': np.float64(0.0)}


In [42]:
import importlib
import GER_CORE.ger_symmetry_test as gst

importlib.reload(gst)

<module 'GER_CORE.ger_symmetry_test' from '/content/GER_CORE/ger_symmetry_test.py'>

In [43]:
result = gst.test_dynamic_symmetry()

print(result)

{'field_difference': np.float64(0.0), 'norm_difference': np.float64(0.0), 'amplitude_difference': np.float64(0.0)}


In [44]:
from GER_CORE.ger_engine import run_engine


for beta in [0.0, 0.1, 1.0, 5.0]:

    result = run_engine(
        n=384,
        timesteps=2000,
        dt=5e-5,
        beta=beta,
        potential="A",
        sigma=0.10,
        snapshot_stride=200
    )


    final = result["snapshots"][-1]


    print("\n======================")
    print("Beta:", beta)
    print("======================")

    print(
        "Energia:",
        result["final"]["energy"]
    )

    print(
        "Erro energia:",
        result["final"]["energy_error"]
    )

    print(
        "Amplitude:",
        max(abs(result["gamma"]))
    )

    print(
        "Modo dominante:",
        final["dominant_mode"]
    )

    print(
        "Entropia:",
        final["spectral_entropy"]
    )

    print(
        "Participação:",
        final["participation_ratio"]
    )

    print(
        "Largura modal:",
        final["modal_width"]
    )


Beta: 0.0
Energia: 0.000188126432190459
Erro energia: 0.00029914052999375785
Amplitude: 0.9998669027438437
Modo dominante: 1
Entropia: 2.7207816148028736
Participação: 13.05224959700054
Largura modal: 8.652540914656656

Beta: 0.1
Energia: -0.00031086391175180804
Erro energia: 0.0011877369310657783
Amplitude: 1.00036746182547
Modo dominante: 1
Entropia: 2.7209535230761652
Participação: 13.05414082021783
Largura modal: 8.654316939708186

Beta: 1.0
Energia: -0.004857305095520187
Erro energia: 0.01223468691520569
Amplitude: 1.0048838011936252
Modo dominante: 1
Entropia: 2.7225032495388497
Participação: 13.071191868299676
Largura modal: 8.670361691053232

Beta: 5.0
Energia: -0.026326957750543074
Erro energia: 0.06389988394057437
Amplitude: 1.0252064256210875
Modo dominante: 1
Entropia: 2.7294439924699008
Participação: 13.14763232104181
Largura modal: 8.742971833386477


In [45]:
from GER_CORE.ger_engine import run_engine


for beta in [10, 20, 50, 100]:

    result = run_engine(
        n=384,
        timesteps=2000,
        dt=2.5e-5,
        beta=beta,
        potential="A",
        sigma=0.10,
        snapshot_stride=200
    )


    final = result["snapshots"][-1]


    print("\n======================")
    print("Beta:", beta)
    print("======================")


    print(
        "Energia:",
        result["final"]["energy"]
    )


    print(
        "Erro energia:",
        result["final"]["energy_error"]
    )


    print(
        "Amplitude:",
        max(abs(result["gamma"]))
    )


    print(
        "Divergiu:",
        result["diverged"]
    )


    print(
        "Modo dominante:",
        final["dominant_mode"]
    )


    print(
        "Entropia:",
        final["spectral_entropy"]
    )


    print(
        "Participação:",
        final["participation_ratio"]
    )


    print(
        "Largura modal:",
        final["modal_width"]
    )


Beta: 10
Energia: -0.0512342363817246
Erro energia: 0.031293207205901404
Amplitude: 1.0125576796766653
Divergiu: False
Modo dominante: 1
Entropia: 2.725191884823526
Participação: 13.100985349065159
Largura modal: 8.698198683404524

Beta: 20
Energia: -0.10591979039925403
Erro energia: 0.06401376597869693
Amplitude: 1.0253087767953546
Divergiu: False
Modo dominante: 1
Entropia: 2.729539689074169
Participação: 13.148890705895242
Largura modal: 8.743829890939358

Beta: 50
Energia: -0.29181277285869994
Erro energia: 0.17122979779679645
Amplitude: 1.0645603267879151
Divergiu: False
Modo dominante: 1
Entropia: 2.7427676919768595
Participação: 13.295169492902753
Largura modal: 8.885520970701506

Beta: 100
Energia: -0.6903484369548653
Erro energia: 0.3848801409210193
Amplitude: 1.1335492274886994
Divergiu: False
Modo dominante: 1
Entropia: 2.7653664934099034
Participação: 13.547794956667975
Largura modal: 9.136858453014757


In [46]:
from GER_CORE.ger_engine import run_engine


for dt in [
    5e-5,
    2.5e-5,
    1.25e-5,
    6.25e-6
]:

    result = run_engine(
        n=384,
        timesteps=2000,
        dt=dt,
        beta=100,
        potential="A",
        sigma=0.10,
        snapshot_stride=200
    )


    final = result["snapshots"][-1]


    print("\n======================")
    print("dt:", dt)
    print("======================")


    print(
        "Energia:",
        result["final"]["energy"]
    )


    print(
        "Erro energia:",
        result["final"]["energy_error"]
    )


    print(
        "Amplitude:",
        max(abs(result["gamma"]))
    )


    print(
        "Modo dominante:",
        final["dominant_mode"]
    )


    print(
        "Entropia:",
        final["spectral_entropy"]
    )


    print(
        "Participação:",
        final["participation_ratio"]
    )


    print(
        "Largura modal:",
        final["modal_width"]
    )


    print(
        "Divergiu:",
        result["diverged"]
    )


dt: 5e-05
Energia: -2.284473885539302
Erro energia: 3.5827909613694597
Amplitude: 1.6786093103466675
Modo dominante: 1
Entropia: 2.9137563102260025
Participação: 15.347211792829915
Largura modal: 11.011698111967966
Divergiu: False

dt: 2.5e-05
Energia: -0.6903484369548653
Erro energia: 0.3848801409210193
Amplitude: 1.1335492274886994
Modo dominante: 1
Entropia: 2.7653664934099034
Participação: 13.547794956667975
Largura modal: 9.136858453014757
Divergiu: False

dt: 1.25e-05
Energia: -0.5388409320041169
Erro energia: 0.08094704403159118
Amplitude: 1.0317712771596868
Modo dominante: 1
Entropia: 2.7317494736795
Participação: 13.17331733895894
Largura modal: 8.767164951874902
Divergiu: False

dt: 6.25e-06
Energia: -0.5081697692088077
Erro energia: 0.019418861133146403
Amplitude: 1.007848939204557
Modo dominante: 1
Entropia: 2.7235999209299564
Participação: 13.083521061803014
Largura modal: 8.68156598326639
Divergiu: False


In [47]:
%%writefile GER_CORE/S26_B21_modal_transfer.py

"""
=========================================================
GER CORE
S26-B.2.1

Transferência Modal

=========================================================

Primeira auditoria espectral da Série B.

Esta rotina utiliza exclusivamente os snapshots produzidos
pelo GER CORE, sem recalcular a base espectral.

Cada snapshot já contém todas as métricas produzidas pelo
Observatório Espectral.

=========================================================
"""

from __future__ import annotations

from GER_CORE.ger_engine import run_engine


# =========================================================
# Auditoria de transferência modal
# =========================================================

def run_modal_transfer(
    n=384,
    beta=1.0,
    sigma=0.10,
    dt=5e-5,
    timesteps=2000,
    snapshot_stride=50,
    potential="A"
):
    """
    Executa uma simulação completa e retorna o histórico
    temporal das métricas espectrais.

    Retorno
    -------

    history : list(dict)

    Cada elemento possui:

        step
        time
        dominant_mode
        entropy
        participation
        modal_center
        modal_width
        modal_energy
        probability
        spectral_bands
    """

    result = run_engine(
        n=n,
        timesteps=timesteps,
        dt=dt,
        beta=beta,
        sigma=sigma,
        potential=potential,
        snapshot_stride=snapshot_stride
    )

    history = []

    for snap in result["snapshots"]:

        history.append({

            "step":
                snap["step"],

            "time":
                snap["time"],

            "dominant_mode":
                snap["dominant_mode"],

            "entropy":
                snap["spectral_entropy"],

            "participation":
                snap["participation_ratio"],

            "modal_center":
                snap["modal_center"],

            "modal_width":
                snap["modal_width"],

            "modal_energy":
                snap["modal_energy"],

            "probability":
                snap["probability"],

            "spectral_bands":
                snap["spectral_bands"]

        })

    return history


# =========================================================
# Impressão resumida
# =========================================================

def print_modal_transfer(history):
    """
    Resumo textual da evolução modal.
    """

    for item in history:

        print(
            f"step: {item['step']:5d} | "
            f"modo: {item['dominant_mode']:3d} | "
            f"entropia: {item['entropy']:.6f} | "
            f"participação: {item['participation']:.6f} | "
            f"largura: {item['modal_width']:.6f}"
        )

Writing GER_CORE/S26_B21_modal_transfer.py


In [48]:
import inspect

from GER_CORE.ger_graph import build_ring_graph, spectral_basis

print(inspect.signature(build_ring_graph))
print(inspect.getsource(build_ring_graph))

(n)
def build_ring_graph(n):
    """
    Constrói o grafo periódico F1.

    Retorna:

    A:
        matriz de adjacência

    L:
        Laplaciano discreto

    theta:
        coordenadas angulares
    """

    A = np.zeros((n, n))

    for i in range(n):

        A[i, (i + 1) % n] = 1.0
        A[i, (i - 1) % n] = 1.0


    D = np.diag(
        np.sum(A, axis=1)
    )


    L = D - A


    theta = np.linspace(
        0.0,
        2.0*np.pi,
        n,
        endpoint=False
    )


    return A, L, theta



In [49]:
result = build_ring_graph(384)

print(type(result))
print(len(result))

for i, item in enumerate(result):
    print(i, type(item), getattr(item, "shape", None))

<class 'tuple'>
3
0 <class 'numpy.ndarray'> (384, 384)
1 <class 'numpy.ndarray'> (384, 384)
2 <class 'numpy.ndarray'> (384,)


In [50]:
from GER_CORE.S26_B21_modal_transfer import (
    run_modal_transfer,
    print_modal_transfer
)

history = run_modal_transfer(
    beta=1.0
)

print_modal_transfer(history)

step:     0 | modo:   1 | entropia: 2.720909 | participação: 13.053917 | largura: 8.653678
step:    50 | modo:   1 | entropia: 2.720911 | participação: 13.053929 | largura: 8.653689
step:   100 | modo:   1 | entropia: 2.720914 | participação: 13.053962 | largura: 8.653721
step:   150 | modo:   1 | entropia: 2.720919 | participação: 13.054017 | largura: 8.653774
step:   200 | modo:   1 | entropia: 2.720926 | participação: 13.054093 | largura: 8.653847
step:   250 | modo:   1 | entropia: 2.720935 | participação: 13.054191 | largura: 8.653941
step:   300 | modo:   1 | entropia: 2.720946 | participação: 13.054310 | largura: 8.654056
step:   350 | modo:   1 | entropia: 2.720959 | participação: 13.054451 | largura: 8.654192
step:   400 | modo:   1 | entropia: 2.720974 | participação: 13.054613 | largura: 8.654348
step:   450 | modo:   1 | entropia: 2.720991 | participação: 13.054797 | largura: 8.654526
step:   500 | modo:   1 | entropia: 2.721010 | participação: 13.055003 | largura: 8.654724

In [51]:
import GER_CORE.S26_B21_modal_transfer as mod

print(dir(mod))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'annotations', 'print_modal_transfer', 'run_engine', 'run_modal_transfer']


In [52]:
import inspect
import GER_CORE.S26_B21_modal_transfer as mod

print(inspect.getsource(mod))


"""
GER CORE
S26-B.2.1

Transferência Modal


Primeira auditoria espectral da Série B.

Esta rotina utiliza exclusivamente os snapshots produzidos
pelo GER CORE, sem recalcular a base espectral.

Cada snapshot já contém todas as métricas produzidas pelo
Observatório Espectral.

"""

from __future__ import annotations

from GER_CORE.ger_engine import run_engine


# =========================================================
# Auditoria de transferência modal
# =========================================================

def run_modal_transfer(
    n=384,
    beta=1.0,
    sigma=0.10,
    dt=5e-5,
    timesteps=2000,
    snapshot_stride=50,
    potential="A"
):
    """
    Executa uma simulação completa e retorna o histórico
    temporal das métricas espectrais.

    Retorno
    -------

    history : list(dict)

    Cada elemento possui:

        step
        time
        dominant_mode
        entropy
        participation
        modal_center
        modal_width
        modal_energy
     

In [53]:
import importlib
import GER_CORE.S26_B21_modal_transfer as mod

importlib.invalidate_caches()
mod = importlib.reload(mod)

print(dir(mod))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'annotations', 'print_modal_transfer', 'run_engine', 'run_modal_transfer']


In [54]:
import GER_CORE.S26_B21_modal_transfer as mod

history = mod.run_modal_transfer(
    beta=1.0
)

mod.print_modal_transfer(history)

step:     0 | modo:   1 | entropia: 2.720909 | participação: 13.053917 | largura: 8.653678
step:    50 | modo:   1 | entropia: 2.720911 | participação: 13.053929 | largura: 8.653689
step:   100 | modo:   1 | entropia: 2.720914 | participação: 13.053962 | largura: 8.653721
step:   150 | modo:   1 | entropia: 2.720919 | participação: 13.054017 | largura: 8.653774
step:   200 | modo:   1 | entropia: 2.720926 | participação: 13.054093 | largura: 8.653847
step:   250 | modo:   1 | entropia: 2.720935 | participação: 13.054191 | largura: 8.653941
step:   300 | modo:   1 | entropia: 2.720946 | participação: 13.054310 | largura: 8.654056
step:   350 | modo:   1 | entropia: 2.720959 | participação: 13.054451 | largura: 8.654192
step:   400 | modo:   1 | entropia: 2.720974 | participação: 13.054613 | largura: 8.654348
step:   450 | modo:   1 | entropia: 2.720991 | participação: 13.054797 | largura: 8.654526
step:   500 | modo:   1 | entropia: 2.721010 | participação: 13.055003 | largura: 8.654724

In [55]:
%%writefile GER_CORE/S26_B22_mixing_rate.py

"""
=========================================================
GER CORE
S26-B.2.2

Taxa de Mistura Espectral

=========================================================

Esta auditoria calcula as taxas temporais das principais
grandezas espectrais produzidas pelo Observatório GER.

Não executa simulações.

Recebe apenas o histórico produzido pela auditoria
S26_B21_modal_transfer.

=========================================================
"""

from __future__ import annotations

import numpy as np


# =========================================================
# Derivada temporal simples
# ==========================================================
def temporal_derivative(values, times):
    """
    Calcula dX/dt por diferenças finitas.
    """

    values = np.asarray(values, dtype=float)
    times = np.asarray(times, dtype=float)

    derivative = np.zeros_like(values)

    derivative[0] = 0.0

    for i in range(1, len(values)):

        dt = times[i] - times[i - 1]

        if dt == 0.0:

            derivative[i] = 0.0

        else:

            derivative[i] = (
                values[i] - values[i - 1]
            ) / dt

    return derivative


# =========================================================
# Auditoria principal
# ==========================================================
def compute_mixing_rate(history):
    """
    Calcula as taxas temporais da mistura espectral.
    """

    time = np.array([
        h["time"]
        for h in history
    ])

    entropy = np.array([
        h["entropy"]
        for h in history
    ])

    width = np.array([
        h["modal_width"]
        for h in history
    ])

    participation = np.array([
        h["participation"]
        for h in history
    ])

    dSdt = temporal_derivative(
        entropy,
        time
    )

    dWdt = temporal_derivative(
        width,
        time
    )

    dPRdt = temporal_derivative(
        participation,
        time
    )

    result = []

    for i in range(len(history)):

        result.append({

            "step":
                history[i]["step"],

            "time":
                history[i]["time"],

            "dSdt":
                dSdt[i],

            "dWdt":
                dWdt[i],

            "dPRdt":
                dPRdt[i]

        })

    return result


# =========================================================
# Impressão resumida
# ==========================================================
def print_mixing_rate(result):

    for r in result:

        print(

            f"step: {r['step']:5d}"

            f" | dS/dt: {r['dSdt']:.6e}"

            f" | dW/dt: {r['dWdt']:.6e}"

            f" | dPR/dt: {r['dPRdt']:.6e}"

        )

Writing GER_CORE/S26_B22_mixing_rate.py


In [56]:
from GER_CORE.S26_B21_modal_transfer import run_modal_transfer
from GER_CORE.S26_B22_mixing_rate import (
    compute_mixing_rate,
    print_mixing_rate
)

history = run_modal_transfer(
    beta=1.0
)

mixing = compute_mixing_rate(
    history
)

print_mixing_rate(mixing)

step:     0 | dS/dt: 0.000000e+00 | dW/dt: 0.000000e+00 | dPR/dt: 0.000000e+00
step:    50 | dS/dt: 4.291032e-04 | dW/dt: 4.482461e-03 | dPR/dt: 4.650972e-03
step:   100 | dS/dt: 1.223745e-03 | dW/dt: 1.278345e-02 | dPR/dt: 1.326395e-02
step:   150 | dS/dt: 2.018403e-03 | dW/dt: 2.108481e-02 | dPR/dt: 2.187710e-02
step:   200 | dS/dt: 2.813089e-03 | dW/dt: 2.938681e-02 | dPR/dt: 3.049056e-02
step:   250 | dS/dt: 3.607814e-03 | dW/dt: 3.768968e-02 | dPR/dt: 3.910443e-02
step:   300 | dS/dt: 4.402588e-03 | dW/dt: 4.599366e-02 | dPR/dt: 4.771883e-02
step:   350 | dS/dt: 5.197424e-03 | dW/dt: 5.429902e-02 | dPR/dt: 5.633389e-02
step:   400 | dS/dt: 5.992330e-03 | dW/dt: 6.260599e-02 | dPR/dt: 6.494972e-02
step:   450 | dS/dt: 6.787320e-03 | dW/dt: 7.091482e-02 | dPR/dt: 7.356644e-02
step:   500 | dS/dt: 7.582403e-03 | dW/dt: 7.922576e-02 | dPR/dt: 8.218417e-02
step:   550 | dS/dt: 8.377589e-03 | dW/dt: 8.753906e-02 | dPR/dt: 9.080302e-02
step:   600 | dS/dt: 9.172891e-03 | dW/dt: 9.585495e

In [57]:
from GER_CORE.ger_engine import run_engine

test = run_engine()

print(type(test))
print(test.keys())

print("\nFINAL:")
print(test["final"].keys())

print("\nSNAPSHOTS:")
print(len(test["snapshots"]))
print(test["snapshots"][-1].keys())

<class 'dict'>
dict_keys(['configuration', 'initial', 'final', 'snapshots', 'gamma', 'laplacian', 'eigenvalues', 'eigenvectors', 'diverged'])

FINAL:
dict_keys(['energy', 'energy_error'])

SNAPSHOTS:
41
dict_keys(['step', 'time', 'energy', 'l2', 'amplitude', 'dominant_mode', 'modal_center', 'modal_width', 'spectral_entropy', 'participation_ratio', 'probability', 'modal_energy', 'spectral_bands', 'gamma', 'energy_error'])


In [58]:
%%writefile GER_CORE/S26_B23_beta_dependence.py

from __future__ import annotations

from GER_CORE.ger_engine import run_engine


def run_beta_dependence(
    betas=None,
    n=256,
    sigma=20,
    dt=0.00025,
    timesteps=2000,
    snapshot_stride=50,
    potential="A"
):

    if betas is None:
        betas = [
            0.1,
            0.25,
            0.5,
            1.0,
            2.0
        ]

    history = []


    for beta in betas:

        simulation = run_engine(
            n=n,
            sigma=sigma,
            dt=dt,
            timesteps=timesteps,
            snapshot_stride=snapshot_stride,
            beta=beta,
            potential=potential
        )


        snapshots = simulation["snapshots"]

        if len(snapshots) == 0:
            raise RuntimeError(
                "Simulação sem snapshots."
            )


        final_snapshot = snapshots[-1]


        result = {

            "beta":
                beta,


            "energy":
                simulation["final"]["energy"],


            "energy_error":
                simulation["final"]["energy_error"],


            "amplitude":
                final_snapshot.get(
                    "amplitude",
                    None
                ),


            "dominant_mode":
                final_snapshot.get(
                    "dominant_mode",
                    None
                ),


            "modal_center":
                final_snapshot.get(
                    "modal_center",
                    None
                ),


            "modal_width":
                final_snapshot.get(
                    "modal_width",
                    None
                ),


            "spectral_entropy":
                final_snapshot.get(
                    "spectral_entropy",
                    None
                ),


            "participation_ratio":
                final_snapshot.get(
                    "participation_ratio",
                    None
                ),


            "spectral_bands":
                final_snapshot.get(
                    "spectral_bands",
                    None
                ),


            "diverged":
                simulation["diverged"]
        }


        history.append(result)


    return history



def print_beta_dependence(history):

    print("\nBETA DEPENDENCE\n")

    for item in history:

        print(
            f"β={item['beta']:.3f} | "
            f"E={item['energy']:.6e} | "
            f"A={item['amplitude']} | "
            f"mode={item['dominant_mode']} | "
            f"PR={item['participation_ratio']}"
        )

Writing GER_CORE/S26_B23_beta_dependence.py


In [59]:
from GER_CORE.S26_B23_beta_dependence import (
    run_beta_dependence,
    print_beta_dependence
)

In [60]:
history = run_beta_dependence()

print_beta_dependence(history)


BETA DEPENDENCE

β=0.100 | E=-2.553385e-02 | A=1.0125911824332876 | mode=0 | PR=1.0000283444172233
β=0.250 | E=-6.758289e-02 | A=1.0317796666273902 | mode=0 | PR=1.0000304352583278
β=0.500 | E=-1.489808e-01 | A=1.0645955740142135 | mode=0 | PR=1.0000340927180966
β=1.000 | E=-3.648762e-01 | A=1.1335868629652142 | mode=0 | PR=1.0000420957610454
β=2.000 | E=-1.127204e+00 | A=1.287066930244913 | mode=0 | PR=1.0000612859371338


In [61]:
import importlib
import GER_CORE.S26_B23_beta_dependence as b23

importlib.reload(b23)

from GER_CORE.S26_B23_beta_dependence import (
    run_beta_dependence,
    print_beta_dependence
)

In [62]:
import inspect
import GER_CORE.S26_B23_beta_dependence as b23

print(inspect.getsource(b23.run_beta_dependence))

def run_beta_dependence(
    betas=None,
    n=256,
    sigma=20,
    dt=0.00025,
    timesteps=2000,
    snapshot_stride=50,
    potential="A"
):

    if betas is None:
        betas = [
            0.1,
            0.25,
            0.5,
            1.0,
            2.0
        ]

    history = []


    for beta in betas:

        simulation = run_engine(
            n=n,
            sigma=sigma,
            dt=dt,
            timesteps=timesteps,
            snapshot_stride=snapshot_stride,
            beta=beta,
            potential=potential
        )


        snapshots = simulation["snapshots"]

        if len(snapshots) == 0:
            raise RuntimeError(
                "Simulação sem snapshots."
            )


        final_snapshot = snapshots[-1]


        result = {

            "beta":
                beta,


            "energy":
                simulation["final"]["energy"],


            "energy_error":
                simulation["final"]["energy_error"],


       

In [63]:
import importlib
import GER_CORE.S26_B23_beta_dependence as b23

importlib.reload(b23)

from GER_CORE.S26_B23_beta_dependence import (
    run_beta_dependence,
    print_beta_dependence
)

In [64]:
from GER_CORE.ger_engine import run_engine

sim = run_engine(
    n=256,
    sigma=20,
    dt=0.00025,
    timesteps=2000,
    snapshot_stride=50,
    beta=0.5
)

print(sim["snapshots"][-1].keys())

dict_keys(['step', 'time', 'energy', 'l2', 'amplitude', 'dominant_mode', 'modal_center', 'modal_width', 'spectral_entropy', 'participation_ratio', 'probability', 'modal_energy', 'spectral_bands', 'gamma', 'energy_error'])


In [65]:
from GER_CORE.S26_B23_beta_dependence import (
    run_beta_dependence,
    print_beta_dependence
)

In [66]:
import importlib
import GER_CORE.S26_B23_beta_dependence as b23

importlib.reload(b23)

from GER_CORE.S26_B23_beta_dependence import (
    run_beta_dependence,
    print_beta_dependence
)

In [67]:
history = run_beta_dependence()

print_beta_dependence(history)


BETA DEPENDENCE

β=0.100 | E=-2.553385e-02 | A=1.0125911824332876 | mode=0 | PR=1.0000283444172233
β=0.250 | E=-6.758289e-02 | A=1.0317796666273902 | mode=0 | PR=1.0000304352583278
β=0.500 | E=-1.489808e-01 | A=1.0645955740142135 | mode=0 | PR=1.0000340927180966
β=1.000 | E=-3.648762e-01 | A=1.1335868629652142 | mode=0 | PR=1.0000420957610454
β=2.000 | E=-1.127204e+00 | A=1.287066930244913 | mode=0 | PR=1.0000612859371338


In [68]:
%%writefile GER_CORE/S26_B23_beta_analysis.py

import numpy as np


def analyze_beta_dependence(history):
    """
    Analisa a resposta dinâmica em função de beta.

    Entrada:
        history:
            lista produzida por run_beta_dependence()

    Saída:
        dicionário com métricas de sensibilidade.
    """

    if len(history) < 2:
        raise ValueError(
            "Histórico insuficiente para análise."
        )


    beta = np.array(
        [
            item["beta"]
            for item in history
        ],
        dtype=float
    )


    energy = np.array(
        [
            item["energy"]
            for item in history
        ],
        dtype=float
    )


    amplitude = np.array(
        [
            item["amplitude"]
            for item in history
        ],
        dtype=float
    )


    modes = [
        item["dominant_mode"]
        for item in history
    ]


    pr = np.array(
        [
            item["participation_ratio"]
            for item in history
        ],
        dtype=float
    )


    dE_dbeta = np.gradient(
        energy,
        beta
    )


    amplitude_gain = (
        amplitude[-1] - amplitude[0]
    ) / amplitude[0]


    mode_stable = (
        len(set(modes)) == 1
    )


    result = {

        "beta_range":
            (
                beta[0],
                beta[-1]
            ),


        "energy_gradient":
            dE_dbeta,


        "energy_monotonic_decrease":
            bool(
                np.all(
                    np.diff(energy) < 0
                )
            ),


        "amplitude_gain":
            amplitude_gain,


        "mode_stable":
            mode_stable,


        "dominant_modes":
            modes,


        "pr_variation":
            (
                pr[-1] - pr[0]
            ),


        "mean_pr":
            np.mean(pr)

    }


    return result



def print_beta_analysis(result):

    print("\nBETA ANALYSIS\n")


    print(
        "Beta range:",
        result["beta_range"]
    )


    print(
        "Energy monotonic decrease:",
        result["energy_monotonic_decrease"]
    )


    print(
        "Amplitude gain:",
        f"{100*result['amplitude_gain']:.3f}%"
    )


    print(
        "Dominant mode stable:",
        result["mode_stable"]
    )


    print(
        "Modes:",
        result["dominant_modes"]
    )


    print(
        "PR variation:",
        result["pr_variation"]
    )


    print(
        "Mean PR:",
        result["mean_pr"]
    )

Writing GER_CORE/S26_B23_beta_analysis.py


In [69]:
from GER_CORE.S26_B23_beta_analysis import (
    analyze_beta_dependence,
    print_beta_analysis
)

In [70]:
analysis = analyze_beta_dependence(history)

print_beta_analysis(analysis)


BETA ANALYSIS

Beta range: (np.float64(0.1), np.float64(2.0))
Energy monotonic decrease: True
Amplitude gain: 27.106%
Dominant mode stable: True
Modes: [0, 0, 0, 0, 0]
PR variation: 3.2941519910512085e-05
Mean PR: 1.0000392508183655


In [71]:
%%writefile GER_CORE/S26_B24_transition_scan.py

import numpy as np

from GER_CORE.ger_engine import run_engine


def run_transition_scan(
    betas=None,
    n=256,
    sigma=20,
    dt=0.00025,
    timesteps=2000,
    snapshot_stride=50,
    potential="A"
):

    if betas is None:
        betas = [
            1,
            2,
            5,
            10,
            20,
            50,
            100
        ]


    history = []


    for beta in betas:

        simulation = run_engine(
            n=n,
            sigma=sigma,
            dt=dt,
            timesteps=timesteps,
            snapshot_stride=snapshot_stride,
            beta=beta,
            potential=potential
        )


        final = simulation["snapshots"][-1]


        history.append({

            "beta":
                beta,

            "energy":
                simulation["final"]["energy"],

            "energy_error":
                simulation["final"]["energy_error"],

            "amplitude":
                final["amplitude"],

            "dominant_mode":
                final["dominant_mode"],

            "spectral_entropy":
                final["spectral_entropy"],

            "participation_ratio":
                final["participation_ratio"],

            "modal_width":
                final["modal_width"],

            "diverged":
                simulation["diverged"]

        })


    return history



def print_transition_scan(history):

    print("\nTRANSITION SCAN\n")


    for item in history:

        print(
            f"β={item['beta']:>6} | "
            f"E={item['energy']:.5e} | "
            f"A={item['amplitude']:.4f} | "
            f"mode={item['dominant_mode']} | "
            f"PR={item['participation_ratio']:.5f} | "
            f"H={item['spectral_entropy']:.4f}"
        )

Writing GER_CORE/S26_B24_transition_scan.py


In [72]:
from GER_CORE.S26_B24_transition_scan import (
    run_transition_scan,
    print_transition_scan
)

In [73]:
transition = run_transition_scan()

print_transition_scan(transition)

/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:127: RuntimeWarning: invalid value encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
/content/GER_CORE/ger_engine.py:197: RuntimeWarning: overflow encountered in multiply
  + beta * force



TRANSITION SCAN

β=     1 | E=-3.64876e-01 | A=1.1336 | mode=0 | PR=1.00004 | H=0.0003
β=     2 | E=-1.12720e+00 | A=1.2871 | mode=0 | PR=1.00006 | H=0.0004
β=     5 | E=-1.29934e+01 | A=1.9369 | mode=0 | PR=1.00016 | H=0.0009
β=    10 | E=-1.23451e+03 | A=5.1972 | mode=0 | PR=1.00116 | H=0.0051
β=    20 | E=nan | A=nan | mode=0 | PR=nan | H=-0.0000
β=    50 | E=nan | A=nan | mode=0 | PR=nan | H=-0.0000
β=   100 | E=nan | A=nan | mode=0 | PR=nan | H=-0.0000


In [74]:
%%writefile GER_CORE/S26_B24_1_temporal_refinement.py

import numpy as np

from GER_CORE.ger_engine import run_engine


def run_temporal_refinement(
    betas=None,
    dts=None,
    n=256,
    sigma=20,
    timesteps=2000,
    snapshot_stride=50,
    potential="A"
):

    if betas is None:
        betas = [
            10,
            20
        ]

    if dts is None:
        dts = [
            2.5e-4,
            1.25e-4,
            6.25e-5,
            3.125e-5
        ]


    history = []


    for beta in betas:

        for dt in dts:

            simulation = run_engine(
                n=n,
                sigma=sigma,
                dt=dt,
                timesteps=timesteps,
                snapshot_stride=snapshot_stride,
                beta=beta,
                potential=potential
            )


            final = simulation["snapshots"][-1]


            history.append({

                "beta":
                    beta,

                "dt":
                    dt,

                "energy":
                    simulation["final"]["energy"],

                "energy_error":
                    simulation["final"]["energy_error"],

                "amplitude":
                    final.get(
                        "amplitude",
                        np.nan
                    ),

                "spectral_entropy":
                    final.get(
                        "spectral_entropy",
                        np.nan
                    ),

                "participation_ratio":
                    final.get(
                        "participation_ratio",
                        np.nan
                    ),

                "diverged":
                    simulation["diverged"]

            })


    return history



def print_temporal_refinement(history):

    print("\nTEMPORAL REFINEMENT\n")

    for item in history:

        print(
            f"β={item['beta']:>3} | "
            f"dt={item['dt']:.2e} | "
            f"E={item['energy']:.5e} | "
            f"A={item['amplitude']} | "
            f"PR={item['participation_ratio']} | "
            f"div={item['diverged']}"
        )

Writing GER_CORE/S26_B24_1_temporal_refinement.py


In [75]:
from GER_CORE.S26_B24_1_temporal_refinement import (
    run_temporal_refinement,
    print_temporal_refinement
)

In [76]:

import importlib

import GER_CORE.ger_graph
import GER_CORE.ger_potential
import GER_CORE.ger_engine

importlib.reload(GER_CORE.ger_graph)
importlib.reload(GER_CORE.ger_potential)
importlib.reload(GER_CORE.ger_engine)

print("GER_CORE recarregado")

GER_CORE recarregado


In [77]:
from GER_CORE.S26_B24_1_temporal_refinement import (
    run_temporal_refinement,
    print_temporal_refinement
)

In [78]:
refinement = run_temporal_refinement()

print_temporal_refinement(refinement)


TEMPORAL REFINEMENT

β= 10 | dt=2.50e-04 | E=-1.23451e+03 | A=5.1972262837036505 | PR=1.0011642120127575 | div=False
β= 10 | dt=1.25e-04 | E=-7.10280e+00 | A=1.3729688330205652 | PR=1.0000728198712556 | div=False
β= 10 | dt=6.25e-05 | E=-3.13135e+00 | A=1.0814113101314713 | PR=1.0000360118018952 | div=False
β= 10 | dt=3.13e-05 | E=-2.60830e+00 | A=1.0197441622794354 | PR=1.0000291258305134 | div=False
β= 20 | dt=2.50e-04 | E=nan | A=nan | PR=nan | div=True
β= 20 | dt=1.25e-04 | E=-5.19736e+01 | A=1.936922086868919 | PR=1.000162273624143 | div=False
β= 20 | dt=6.25e-05 | E=-8.10627e+00 | A=1.1699007160420125 | PR=1.0000464762248011 | div=False
β= 20 | dt=3.13e-05 | E=-5.53836e+00 | A=1.0398843782247231 | PR=1.000031335626599 | div=False


In [79]:
%%writefile GER_CORE/S26_B24_2_stability_map.py

import numpy as np

from GER_CORE.ger_engine import run_engine


def run_stability_map(
    betas=None,
    dts=None,
    n=256,
    sigma=20,
    timesteps=2000,
    snapshot_stride=50,
    potential="A"
):

    if betas is None:
        betas = [
            1,
            2,
            5,
            10,
            20,
            30,
            50
        ]


    if dts is None:
        dts = [
            2.5e-4,
            1.25e-4,
            6.25e-5,
            3.125e-5
        ]


    stability = []


    for beta in betas:

        for dt in dts:

            try:

                simulation = run_engine(
                    n=n,
                    sigma=sigma,
                    dt=dt,
                    timesteps=timesteps,
                    beta=beta,
                    snapshot_stride=snapshot_stride,
                    potential=potential
                )


                snapshots = simulation["snapshots"]

                if len(snapshots) == 0:
                    raise RuntimeError(
                        "Sem snapshots"
                    )


                final_snapshot = snapshots[-1]


                result = {

                    "beta":
                        beta,

                    "dt":
                        dt,

                    "energy":
                        simulation["final"]["energy"],

                    "energy_error":
                        simulation["final"]["energy_error"],

                    "amplitude":
                        final_snapshot.get(
                            "amplitude",
                            np.nan
                        ),

                    "participation_ratio":
                        final_snapshot.get(
                            "participation_ratio",
                            np.nan
                        ),

                    "dominant_mode":
                        final_snapshot.get(
                            "dominant_mode",
                            np.nan
                        ),

                    "stable":
                        not simulation["diverged"]
                    and
                        np.isfinite(
                            simulation["final"]["energy"]
                        )

                }


            except Exception as e:

                result = {

                    "beta":
                        beta,

                    "dt":
                        dt,

                    "energy":
                        np.nan,

                    "energy_error":
                        np.nan,

                    "amplitude":
                        np.nan,

                    "participation_ratio":
                        np.nan,

                    "dominant_mode":
                        np.nan,

                    "stable":
                        False,

                    "error":
                        str(e)
                }


            stability.append(result)


    return stability



def print_stability_map(stability):

    print("\nSTABILITY MAP\n")

    for item in stability:

        status = "YES" if item["stable"] else "NO"

        print(
            f"β={item['beta']:>5} | "
            f"dt={item['dt']:.2e} | "
            f"stable={status:<3} | "
            f"E={item['energy']:.5e} | "
            f"A={item['amplitude']:.5f}"
        )

Writing GER_CORE/S26_B24_2_stability_map.py


In [80]:
from GER_CORE.S26_B24_2_stability_map import (
    run_stability_map,
    print_stability_map
)

In [81]:
stability = run_stability_map()

print_stability_map(stability)


STABILITY MAP

β=    1 | dt=2.50e-04 | stable=YES | E=-3.64876e-01 | A=1.13359
β=    1 | dt=1.25e-04 | stable=YES | E=-2.70332e-01 | A=1.03178
β=    1 | dt=6.25e-05 | stable=YES | E=-2.51762e-01 | A=1.00785
β=    1 | dt=3.13e-05 | stable=YES | E=-2.47384e-01 | A=1.00196
β=    2 | dt=2.50e-04 | stable=YES | E=-1.12720e+00 | A=1.28707
β=    2 | dt=1.25e-04 | stable=YES | E=-5.95923e-01 | A=1.06460
β=    2 | dt=6.25e-05 | stable=YES | E=-5.15520e-01 | A=1.01576
β=    2 | dt=3.13e-05 | stable=YES | E=-4.97665e-01 | A=1.00392
β=    5 | dt=2.50e-04 | stable=YES | E=-1.29934e+01 | A=1.93692
β=    5 | dt=1.25e-04 | stable=YES | E=-2.02657e+00 | A=1.16990
β=    5 | dt=6.25e-05 | stable=YES | E=-1.38459e+00 | A=1.03988
β=    5 | dt=3.13e-05 | stable=YES | E=-1.26622e+00 | A=1.00982
β=   10 | dt=2.50e-04 | stable=YES | E=-1.23451e+03 | A=5.19723
β=   10 | dt=1.25e-04 | stable=YES | E=-7.10280e+00 | A=1.37297
β=   10 | dt=6.25e-05 | stable=YES | E=-3.13135e+00 | A=1.08141
β=   10 | dt=3.13e-05 | 

In [82]:
%%writefile GER_CORE/S26_B24_3_critical_curve.py

import numpy as np

from GER_CORE.ger_engine import run_engine


def find_critical_dt(
    beta,
    dts=None,
    n=256,
    sigma=20,
    timesteps=2000,
    snapshot_stride=50,
    potential="A"
):

    if dts is None:
        dts = [
            5e-4,
            2.5e-4,
            1.25e-4,
            6.25e-5,
            3.125e-5,
            1.5625e-5
        ]


    results = []


    for dt in dts:

        try:

            simulation = run_engine(
                n=n,
                sigma=sigma,
                dt=dt,
                timesteps=timesteps,
                beta=beta,
                snapshot_stride=snapshot_stride,
                potential=potential
            )


            snapshot = simulation["snapshots"][-1]


            stable = (
                not simulation["diverged"]
                and
                np.isfinite(
                    simulation["final"]["energy"]
                )
            )


            results.append(
                {
                    "beta": beta,
                    "dt": dt,
                    "stable": stable,
                    "energy":
                        simulation["final"]["energy"],
                    "amplitude":
                        snapshot.get(
                            "amplitude",
                            np.nan
                        )
                }
            )


        except Exception:

            results.append(
                {
                    "beta": beta,
                    "dt": dt,
                    "stable": False,
                    "energy": np.nan,
                    "amplitude": np.nan
                }
            )


    stable_dts = [
        r["dt"]
        for r in results
        if r["stable"]
    ]


    critical = None

    if len(stable_dts) > 0:
        critical = max(stable_dts)


    return {
        "beta": beta,
        "critical_dt": critical,
        "scan": results
    }



def run_critical_curve(
    betas=None
):

    if betas is None:
        betas = [
            1,
            2,
            5,
            10,
            20,
            30,
            50
        ]


    curve = []


    for beta in betas:

        result = find_critical_dt(
            beta
        )

        curve.append(
            result
        )


    return curve



def print_critical_curve(curve):

    print("\nCRITICAL DT CURVE\n")

    for item in curve:

        print(
            f"β={item['beta']:>5} | "
            f"dt_crit={item['critical_dt']}"
        )

Writing GER_CORE/S26_B24_3_critical_curve.py


In [83]:
from GER_CORE.S26_B24_3_critical_curve import (
    run_critical_curve,
    print_critical_curve
)

In [84]:
curve = run_critical_curve()

print_critical_curve(curve)


CRITICAL DT CURVE

β=    1 | dt_crit=0.0005
β=    2 | dt_crit=0.0005
β=    5 | dt_crit=0.00025
β=   10 | dt_crit=0.00025
β=   20 | dt_crit=0.000125
β=   30 | dt_crit=0.000125
β=   50 | dt_crit=0.000125
